# Độ ổn định theo seed của các nhánh Edge chính (R1: "multiple seeds")

**Chạy SAU notebook Sleep-EDF** (cần `runs/seed42/T_ce`, `runs/seed42/A_kd_Tce` trên Drive). GPU L4 là đủ.

- Chạy lại **3 nhánh Edge**: `E_scratch` (chỉ học từ nhãn), `E_direct_Tce` (distill trực tiếp từ Teacher-CE),
  `E_staged_Tce` (distill từ Assistant-KD (Teacher-CE), model Edge chính) với **seed huấn luyện 43 và 44** (chốt trước khi xem kết quả).
- Nguồn distill **cố định = model seed 42 của cùng fold** (`src_seed = 42`): chỉ khởi tạo và thứ tự batch của Edge thay đổi;
  chia fold không đổi (`SPLIT_SEED` = 42) → không rò rỉ dữ liệu. Kết quả là độ ổn định của Edge **khi Teacher/Assistant cố định**.
- Giao thức train giống hệt seed 42 (cùng `TRAIN_CFG`, T = 4, trọng số KD).
- Chạy song song 2 phiên Colab: phiên 1 đặt `RUN_SEEDS = [43]`, phiên 2 `RUN_SEEDS = [44]`; mỗi run resume được.

**Phân tích (chốt trước):** 3 so sánh Macro-F1 — Staged − Direct, Staged − Scratch, Direct − Scratch (Teacher-CE) — trên dữ liệu gộp
3 seed (42, 43, 44), bootstrap theo cụm subject, BH trong 3 so sánh; accuracy, κ, N1-F1 là family phụ. Vì cả 3 so sánh đều có nhánh
thuộc dòng Teacher-CE, dùng **79 subject** (bỏ nhóm F1, quy tắc lai); bản 100 subject là độ nhạy. Kèm kết quả từng seed riêng.

In [ ]:
# ===================== CONFIG =====================
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT = '/content/drive/MyDrive/sleep_kd_r2'
CODE_DIR   = '/content/skd_code'
# Sleep-EDF
RAW_DIR    = '/content/sleep-edfx'                 # EDF gốc (local, tải lại được)
DATA_DIR   = f'{DRIVE_ROOT}/data_sleepedf'         # npz theo từng recording (lưu Drive)
LOCAL_DATA = '/content/data_sleepedf'              # bản copy local để đọc nhanh
META_CSV   = f'{DRIVE_ROOT}/recordings.csv'
FOLDS_JSON = f'{DRIVE_ROOT}/folds_subject_level.json'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
RESULTS    = f'{DRIVE_ROOT}/results'
for d in [DRIVE_ROOT, DATA_DIR, RUNS_DIR, RESULTS, f'{CODE_DIR}/model_v11']:
    os.makedirs(d, exist_ok=True)

# ---- Quyết định giao thức (ghi rõ trong Methods)
K_FOLDS          = 5
SPLIT_SEED       = 42    # seed CHIA FOLD (cố định cho mọi phân tích) — khác seed huấn luyện ở SEEDS
TRIM_WAKE_MIN    = 30    # epoch mục tiêu = epoch có nhãn trong [epoch ngủ đầu − 30 phút, epoch ngủ cuối + 30 phút]
STORE_FULL_NIGHT = False # True: lưu cả đêm để báo cáo thêm kết quả không cắt Wake
REJECT_PTP_UV    = None  # None = không loại epoch nào

# ---- Huấn luyện
SEEDS     = [42]         # seed HUẤN LUYỆN (khởi tạo + thứ tự batch); fold giữ nguyên theo SPLIT_SEED
RUN_FOLDS = [0, 1, 2, 3, 4]
RUN_ARMS  = None         # None = tất cả; hoặc list tên arm để chạy một phần
TRAIN_CFG = dict(EPOCHS=80, PATIENCE=20, LR=1e-3, WD=1e-4, LABEL_SMOOTHING=0.05, FOCAL_GAMMA=2.0,
                 W_CE=0.3, W_KL=0.5, W_MSE=0.2, EVAL_BS=256, NUM_WORKERS=min(8, os.cpu_count()), USE_BF16=True)

SEED_ARMS = [
    dict(name='E_scratch',    kind='edge', loss='ce', bs=64),
    dict(name='E_direct_Tce', kind='edge', loss='kd', src='T_ce',     T=4.0, bs=64, src_seed=42),
    dict(name='E_staged_Tce', kind='edge', loss='kd', src='A_kd_Tce', T=4.0, bs=64, src_seed=42),
]
NEW_SEEDS = [43, 44]
RUN_SEEDS = [43, 44]          # chạy song song: phiên 1 = [43], phiên 2 = [44]
SEED_CONTRASTS = [('E_staged_Tce', 'E_direct_Tce'), ('E_staged_Tce', 'E_scratch'), ('E_direct_Tce', 'E_scratch')]
C_NEW = dict(TRAIN_CFG, K_FOLDS=K_FOLDS, RUNS_DIR=RUNS_DIR, ARMS=SEED_ARMS, SEEDS=RUN_SEEDS)
print('config OK')

In [ ]:
!pip -q install mne thop

## 0. Code (model gốc của project + module chung)
Các file `model_v11/*` và `losses_v11.py` được chép nguyên văn từ project. Thay đổi duy nhất: `StudentSleepNetV11` và `StudentSleepNetV2` trả thêm `cnn_feat`/`ctx_feat` (feature gốc) để làm model nguồn cho KD.

In [ ]:
%%writefile /content/skd_code/model_v11/__init__.py
# -*- coding: utf-8 -*-
"""
model_v11: Teacher-Student Architecture for Sleep Staging
  - Teacher: Multi-branch CNN (EOG/EEG) + Standard Transformer
  - Student:  Light Multi-branch CNN + Linear Attention (O(N))
"""
from .teacher_v11 import TeacherSleepNetV11
from .student_v11 import StudentSleepNetV11

In [ ]:
%%writefile /content/skd_code/model_v11/blocks_v11.py
# -*- coding: utf-8 -*-
"""
model_v11/blocks_v11.py
Các khối xây dựng cho kiến trúc v11:

  1. EogBranch       : CNN nhánh EOG (kernel lớn ~50-100) — bắt sóng cuộn mắt chậm (SEM)
  2. EegBranch       : CNN nhánh EEG (kernel nhỏ ~15-30)  — bắt alpha dropout sắc nét
  3. MultiBranchCNN  : Ghép EOG+EEG → Linear → Embedding (Teacher-size)
  4. LightMultiBranchCNN : Phiên bản nhẹ cho Student
  5. LinearAttention : Khối O(N) Kernel Attention (xấp xỉ Softmax không có ma trận NxN)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ─────────────────────────────────────────────────────────────
# 1. EogBranch — Kernel lớn, bắt sóng lượn chậm của mắt (EOG)
# ─────────────────────────────────────────────────────────────

class EogBranch(nn.Module):
    """
    Xử lý kênh EOG (1 kênh, index [2:3] trong tensor 3 kênh).
    Dùng Conv1d với kernel lớn để bắt Slow Eye Movements (SEM) —
    tín hiệu tần số thấp, biên độ rộng, đặc trưng của giai đoạn N1.

    Input:  (B, 1, T)
    Output: (B, out_ch, T')
    """
    def __init__(self, out_ch: int = 64, kernel_sizes=(51, 101),
                 stride: int = 5, dropout: float = 0.1):
        super().__init__()
        # Multi-scale Inception: 2 nhánh kernel lớn song song
        self.branches = nn.ModuleList()
        for k in kernel_sizes:
            self.branches.append(nn.Sequential(
                nn.Conv1d(1, out_ch // 2, kernel_size=k, stride=stride,
                          padding=k // 2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=out_ch // 2),
                nn.GELU(),
            ))
        # Fusion: (out_ch//2 * 2) → out_ch
        self.fusion = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 1, T) → (B, out_ch, T')"""
        parts = []
        for branch in self.branches:
            out = branch(x)
            parts.append(out)
        # Cắt về cùng chiều dài (do padding lẻ có thể lệch 1)
        min_len = min(p.shape[-1] for p in parts)
        parts = [p[..., :min_len] for p in parts]
        x_cat = torch.cat(parts, dim=1)   # (B, out_ch, T')
        return self.fusion(x_cat)


# ─────────────────────────────────────────────────────────────
# 2. EegBranch — Kernel nhỏ, bắt rớt Alpha sắc nét (EEG)
# ─────────────────────────────────────────────────────────────

class EegBranch(nn.Module):
    """
    Xử lý 2 kênh EEG (index [0:2] trong tensor 3 kênh).
    Dùng Conv1d với kernel nhỏ để bắt alpha dropout đột ngột
    và các gợn sóng nhanh đặc trưng của vùng chẩm khi chuyển sang N1.

    Input:  (B, 2, T)
    Output: (B, out_ch, T')
    """
    def __init__(self, out_ch: int = 64, kernel_sizes=(15, 31),
                 stride: int = 5, dropout: float = 0.1):
        super().__init__()
        self.branches = nn.ModuleList()
        for k in kernel_sizes:
            self.branches.append(nn.Sequential(
                nn.Conv1d(2, out_ch // 2, kernel_size=k, stride=stride,
                          padding=k // 2, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=out_ch // 2),
                nn.GELU(),
            ))
        self.fusion = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, kernel_size=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_ch),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 2, T) → (B, out_ch, T')"""
        parts = []
        for branch in self.branches:
            out = branch(x)
            parts.append(out)
        min_len = min(p.shape[-1] for p in parts)
        parts = [p[..., :min_len] for p in parts]
        x_cat = torch.cat(parts, dim=1)
        return self.fusion(x_cat)


# ─────────────────────────────────────────────────────────────
# 3. MultiBranchCNN — Teacher-size, embed_dim 256
# ─────────────────────────────────────────────────────────────

class MultiBranchCNN(nn.Module):
    """
    Tách 3 kênh → 2 nhánh xử lý riêng biệt:
      - Nhánh EOG: kênh index 2 (1 channel)
      - Nhánh EEG: kênh index 0,1 (2 channels)
    Ghép (Concat) output, qua Linear → vector embedding (embed_dim).

    Input:  (B, 3, T=3000)
    Output: (B, embed_dim)
    """
    def __init__(self, eog_out=64, eeg_out=64, embed_dim=256,
                 stride=5, dropout=0.1):
        super().__init__()
        self.eog_branch = EogBranch(out_ch=eog_out, stride=stride, dropout=dropout)
        self.eeg_branch = EegBranch(out_ch=eeg_out, stride=stride, dropout=dropout)

        # Pooling + Projection: (eog_out + eeg_out) → embed_dim
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(
            nn.Linear(eog_out + eeg_out, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, T) → (B, embed_dim)"""
        eeg = self.eeg_branch(x[:, :2, :])   # (B, eeg_out, T')
        eog = self.eog_branch(x[:, 2:3, :])  # (B, eog_out, T')

        # Cắt về cùng T' nếu lệch
        min_t = min(eeg.shape[-1], eog.shape[-1])
        eeg, eog = eeg[..., :min_t], eog[..., :min_t]

        merged = torch.cat([eeg, eog], dim=1)  # (B, eog_out+eeg_out, T')
        merged = self.pool(merged).squeeze(-1)  # (B, eog_out+eeg_out)
        return self.proj(merged)                # (B, embed_dim)


# ─────────────────────────────────────────────────────────────
# 4. LightMultiBranchCNN — Student-size, embed_dim 128
# ─────────────────────────────────────────────────────────────

class LightMultiBranchCNN(nn.Module):
    """
    Phiên bản CNN nhẹ cho Student.
    Giữ triết lý 2-nhánh, giảm số filters và channels.

    Input:  (B, 3, T=3000)
    Output: (B, embed_dim)
    """
    def __init__(self, eog_out=32, eeg_out=32, embed_dim=128,
                 stride=5, dropout=0.1):
        super().__init__()
        # Chỉ dùng 1 kernel mỗi nhánh (đơn giản hơn Teacher)
        self.eog_conv = nn.Sequential(
            nn.Conv1d(1, eog_out, kernel_size=51, stride=stride,
                      padding=25, bias=False),
            nn.GroupNorm(num_groups=4, num_channels=eog_out),
            nn.GELU(),
        )
        self.eeg_conv = nn.Sequential(
            nn.Conv1d(2, eeg_out, kernel_size=15, stride=stride,
                      padding=7, bias=False),
            nn.GroupNorm(num_groups=4, num_channels=eeg_out),
            nn.GELU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(
            nn.Linear(eog_out + eeg_out, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        eeg = self.eeg_conv(x[:, :2, :])
        eog = self.eog_conv(x[:, 2:3, :])
        min_t = min(eeg.shape[-1], eog.shape[-1])
        eeg, eog = eeg[..., :min_t], eog[..., :min_t]
        merged = torch.cat([eeg, eog], dim=1)
        merged = self.pool(merged).squeeze(-1)
        return self.proj(merged)


# ─────────────────────────────────────────────────────────────
# 5. LinearAttention — O(N) Kernel Attention (không có NxN matrix)
# ─────────────────────────────────────────────────────────────

class LinearAttentionLayer(nn.Module):
    """
    Kernel-based Linear Attention theo Katharopoulos et al. (2020):
    "Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention"

    Xấp xỉ Attention(Q, K, V) = phi(Q) @ (phi(K)^T @ V)
    với phi(x) = elu(x) + 1 (đảm bảo dương).

    Độ phức tạp: O(N * d^2) thay vì O(N^2 * d).

    Args:
        embed_dim: Chiều vector embedding.
        num_heads: Số đầu Attention.
        dropout:   Tỷ lệ dropout.
    """
    def __init__(self, embed_dim: int = 128, num_heads: int = 4,
                 dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) phải chia hết cho num_heads ({num_heads})"
        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads
        self.scale      = self.head_dim ** 0.5

        self.qkv_proj   = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.out_proj   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout    = nn.Dropout(dropout)
        self.norm       = nn.LayerNorm(embed_dim)

    @staticmethod
    def _phi(x: torch.Tensor) -> torch.Tensor:
        """Feature map phi(x) = elu(x) + 1. Đảm bảo kết quả dương."""
        return F.elu(x) + 1.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, N, D) — B=batch, N=seq_len, D=embed_dim
        returns: (B, N, D)
        """
        B, N, D = x.shape
        H, Hd   = self.num_heads, self.head_dim

        # Project Q, K, V
        qkv = self.qkv_proj(x)                  # (B, N, 3*D)
        q, k, v = qkv.chunk(3, dim=-1)           # 3x (B, N, D)

        # Reshape thành multi-head: (B, H, N, Hd)
        q = q.view(B, N, H, Hd).permute(0, 2, 1, 3)
        k = k.view(B, N, H, Hd).permute(0, 2, 1, 3)
        v = v.view(B, N, H, Hd).permute(0, 2, 1, 3)

        # Áp dụng feature map phi
        q = self._phi(q)                         # (B, H, N, Hd)
        k = self._phi(k)                         # (B, H, N, Hd)

        # Linear Attention:
        # S = phi(K)^T @ V → (B, H, Hd, Hd)   [Không phụ thuộc N!]
        # Z = phi(Q) @ sum(phi(K), dim=N)^T     [Normalizer]
        kv = torch.einsum('bhnd,bhnm->bhdm', k, v)  # (B, H, Hd, Hd)
        # Phân tử: phi(Q) @ kv → (B, H, N, Hd)
        qkv_out = torch.einsum('bhnd,bhdm->bhnm', q, kv)

        # Chuẩn hóa (chia cho tổng phi(K))
        k_sum = k.sum(dim=2, keepdim=True)           # (B, H, 1, Hd)
        denom = torch.einsum('bhnd,bhd->bhn', q, k_sum.squeeze(2)) + 1e-6
        denom = denom.unsqueeze(-1)                  # (B, H, N, 1)
        attn_out = qkv_out / denom                   # (B, H, N, Hd)

        # Merge heads
        attn_out = attn_out.permute(0, 2, 1, 3).contiguous().view(B, N, D)
        out = self.out_proj(self.dropout(attn_out))
        return self.norm(x + out)                    # Residual + Norm


class LinearAttentionBlock(nn.Module):
    """
    1 block hoàn chỉnh: LinearAttention + Feed-Forward Network.
    Giống TransformerEncoderLayer nhưng thay Self-Attention bằng Linear Attention.
    """
    def __init__(self, embed_dim: int = 128, num_heads: int = 4,
                 ff_mult: int = 2, dropout: float = 0.1):
        super().__init__()
        self.attn = LinearAttentionLayer(embed_dim, num_heads, dropout)
        self.ffn  = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * ff_mult, embed_dim),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, N, D) → (B, N, D)"""
        x = self.attn(x)
        return self.norm2(x + self.ffn(x))

In [ ]:
%%writefile /content/skd_code/model_v11/teacher_v11.py
# -*- coding: utf-8 -*-
"""
model_v11/teacher_v11.py
Teacher Model cho Knowledge Distillation v11.

Kiến trúc:
  1. MultiBranchCNN: Tách EOG (kernel lớn) / EEG (kernel nhỏ) → embed_dim=256
  2. TransformerEncoder: Standard O(N²) Self-Attention để học context seq dài
  3. Classifier: LayerNorm → Linear → GELU → Dropout → Linear(5)

Input:  (B, L, 3, 3000) — L = seq_len (e.g. 5), 3 kênh, 3000 samples/epoch
Output: dict {'logits': (B, 5), 'cnn_feat': (B, embed_dim),
               'ctx_feat': (B, embed_dim)}
"""

import torch
import torch.nn as nn
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from model_v11.blocks_v11 import MultiBranchCNN


class TeacherSleepNetV11(nn.Module):
    """
    Teacher Model v11.
    ~400K–600K parameters (tùy embed_dim và Transformer config).

    Args:
        embed_dim:    Chiều embedding (output của MultiBranchCNN). Mặc định 256.
        num_heads:    Số đầu Attention của Transformer. Mặc định 8.
        num_layers:   Số lớp TransformerEncoder. Mặc định 3.
        ff_mult:      Hệ số FFN (dim * ff_mult). Mặc định 4.
        num_classes:  Số nhãn phân loại (5 giai đoạn ngủ). Mặc định 5.
        seq_len:      Độ dài chuỗi epoch. Mặc định 5.
        dropout:      Tỷ lệ dropout. Mặc định 0.1.
    """

    def __init__(self,
                 embed_dim: int = 256,
                 num_heads: int = 8,
                 num_layers: int = 3,
                 ff_mult: int = 4,
                 num_classes: int = 5,
                 seq_len: int = 5,
                 dropout: float = 0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.seq_len   = seq_len

        # --- Khối 1: Multi-branch CNN (EOG/EEG) ---
        self.feature_extractor = MultiBranchCNN(
            eog_out=embed_dim // 2,
            eeg_out=embed_dim // 2,
            embed_dim=embed_dim,
            stride=5,
            dropout=dropout,
        )

        # --- Positional Embedding (Learnable) ---
        self.pos_emb = nn.Embedding(seq_len, embed_dim)

        # --- Khối 2: Standard Transformer Encoder (O(N²)) ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * ff_mult,
            dropout=dropout,
            activation='gelu',
            batch_first=True,   # (B, N, D) convention
            norm_first=True,    # Pre-LayerNorm (Pre-LN) — ổn định hơn
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(embed_dim),
        )

        # --- Khối 3: Classifier ---
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> dict:
        """
        x: (B, L, 3, T)
        returns: dict {
            'logits'   : (B, num_classes),
            'cnn_feat' : (B, embed_dim)  — output của CNN, dùng cho Feature Matching,
            'ctx_feat' : (B, embed_dim)  — output của Transformer, dùng cho Context Matching
        }
        """
        B, L, C, T = x.shape

        # ① Trích đặc trưng từng epoch song song
        x_flat  = x.reshape(B * L, C, T)          # (B*L, 3, T)
        feats   = self.feature_extractor(x_flat)   # (B*L, embed_dim)
        feats   = feats.view(B, L, self.embed_dim) # (B, L, embed_dim)

        # ② Lưu CNN feature của epoch trung tâm (center epoch) để distill
        center  = L // 2
        cnn_feat = feats[:, center, :]             # (B, embed_dim)

        # ③ Positional Embedding
        pos_ids = torch.arange(L, device=x.device)
        feats   = feats + self.pos_emb(pos_ids)    # (B, L, embed_dim)

        # ④ Transformer context
        ctx_seq = self.transformer(feats)           # (B, L, embed_dim)
        ctx_feat = ctx_seq[:, center, :]            # (B, embed_dim) — lấy epoch trung tâm

        # ⑤ Classifier
        logits = self.classifier(ctx_feat)          # (B, num_classes)

        return {
            'logits'  : logits,
            'cnn_feat': cnn_feat,
            'ctx_feat': ctx_feat,
        }

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


if __name__ == '__main__':
    model = TeacherSleepNetV11()
    x = torch.randn(4, 5, 3, 3000)
    out = model(x)
    print(f'TeacherSleepNetV11 | Params: {model.count_parameters():,}')
    print(f'Logits: {out["logits"].shape} | CNN feat: {out["cnn_feat"].shape} | CTX feat: {out["ctx_feat"].shape}')

In [ ]:
%%writefile /content/skd_code/model_v11/student_v11.py
# -*- coding: utf-8 -*-
"""
model_v11/student_v11.py
Student Model siêu nhẹ cho Edge AI — Knowledge Distillation v11.

Kiến trúc:
  1. LightMultiBranchCNN: Tách EOG/EEG → embed_dim=128 (nhẹ hơn Teacher)
  2. LinearAttentionBlock: O(N) thay thế Transformer — không có ma trận NxN
  3. AdaptorLayer: Projection để khớp feature space với Teacher (cho MSE Loss)
  4. Classifier: LayerNorm → Linear(5)

Input:  (B, L, 3, 3000)
Output: dict {'logits': (B, 5), 'cnn_feat_proj': (B, teacher_embed_dim),
               'ctx_feat_proj': (B, teacher_embed_dim)}
"""

import torch
import torch.nn as nn
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from model_v11.blocks_v11 import LightMultiBranchCNN, LinearAttentionBlock


class StudentSleepNetV11(nn.Module):
    """
    Student Model v11 — Tối ưu cho Edge AI.
    ~50K–100K parameters.

    Args:
        embed_dim:          Chiều embedding Student. Mặc định 128.
        teacher_embed_dim:  Chiều embedding Teacher (dùng cho Adaptor). Mặc định 256.
        num_heads:          Số đầu Linear Attention. Mặc định 4.
        num_layers:         Số lớp Linear Attention. Mặc định 2.
        num_classes:        Số nhãn. Mặc định 5.
        seq_len:            Độ dài chuỗi. Mặc định 5.
        dropout:            Dropout ratio. Mặc định 0.1.
    """

    def __init__(self,
                 embed_dim: int = 128,
                 teacher_embed_dim: int = 256,
                 num_heads: int = 4,
                 num_layers: int = 2,
                 num_classes: int = 5,
                 seq_len: int = 5,
                 dropout: float = 0.1):
        super().__init__()
        self.embed_dim         = embed_dim
        self.teacher_embed_dim = teacher_embed_dim
        self.seq_len           = seq_len

        # --- Khối 1: Light Multi-branch CNN ---
        self.feature_extractor = LightMultiBranchCNN(
            eog_out=embed_dim // 2,
            eeg_out=embed_dim // 2,
            embed_dim=embed_dim,
            stride=5,
            dropout=dropout,
        )

        # --- Positional Embedding ---
        self.pos_emb = nn.Embedding(seq_len, embed_dim)

        # --- Khối 2: Linear Attention O(N) (thay Transformer) ---
        self.linear_attn_layers = nn.ModuleList([
            LinearAttentionBlock(embed_dim, num_heads, ff_mult=2, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.ctx_norm = nn.LayerNorm(embed_dim)

        # --- Khối 3: Adaptor Layer (Student → Teacher space) ---
        # Dùng cho Feature Matching Loss (MSE giữa Student và Teacher)
        # Tránh ép khớp trực tiếp 2 không gian khác nhau
        self.cnn_adaptor = nn.Linear(embed_dim, teacher_embed_dim)
        self.ctx_adaptor = nn.Linear(embed_dim, teacher_embed_dim)

        # --- Khối 4: Classifier ---
        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> dict:
        """
        x: (B, L, 3, T)
        returns: dict {
            'logits'         : (B, num_classes),
            'cnn_feat_proj'  : (B, teacher_embed_dim) — để MSE với teacher cnn_feat,
            'ctx_feat_proj'  : (B, teacher_embed_dim) — để MSE với teacher ctx_feat
        }
        """
        B, L, C, T = x.shape

        # ① CNN Feature extraction
        x_flat = x.reshape(B * L, C, T)           # (B*L, 3, T)
        feats  = self.feature_extractor(x_flat)    # (B*L, embed_dim)
        feats  = feats.view(B, L, self.embed_dim)  # (B, L, embed_dim)

        # ② CNN feat (epoch trung tâm) để feature matching
        center   = L // 2
        cnn_feat = feats[:, center, :]             # (B, embed_dim)

        # ③ Positional Embedding
        pos_ids = torch.arange(L, device=x.device)
        feats   = feats + self.pos_emb(pos_ids)    # (B, L, embed_dim)

        # ④ Linear Attention O(N) — xử lý toàn bộ sequence
        ctx = feats
        for layer in self.linear_attn_layers:
            ctx = layer(ctx)                       # (B, L, embed_dim)
        ctx_seq = self.ctx_norm(ctx)
        ctx_feat = ctx_seq[:, center, :]           # (B, embed_dim)

        # ⑤ Adaptor projection (để dùng cho Feature Matching Loss)
        cnn_feat_proj = self.cnn_adaptor(cnn_feat) # (B, teacher_embed_dim)
        ctx_feat_proj = self.ctx_adaptor(ctx_feat) # (B, teacher_embed_dim)

        # ⑥ Classifier
        logits = self.classifier(ctx_feat)         # (B, num_classes)

        return {
            'logits'        : logits,
            'cnn_feat'      : cnn_feat,       # native features (used when this model is a KD source)
            'ctx_feat'      : ctx_feat,
            'cnn_feat_proj' : cnn_feat_proj,
            'ctx_feat_proj' : ctx_feat_proj,
        }

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


if __name__ == '__main__':
    model = StudentSleepNetV11()
    x = torch.randn(4, 5, 3, 3000)
    out = model(x)
    print(f'StudentSleepNetV11 | Params: {model.count_parameters():,}')
    print(f'Logits:       {out["logits"].shape}')
    print(f'CNN feat proj:{out["cnn_feat_proj"].shape}')
    print(f'CTX feat proj:{out["ctx_feat_proj"].shape}')

In [ ]:
%%writefile /content/skd_code/model_v11/student_v2.py
# -*- coding: utf-8 -*-
"""
model_v11/student_v2.py
Student V2 — Siêu nhỏ cho Edge AI / MCU / Smartwatch

Kiến trúc:
  1. DWSep Multi-branch CNN (Depthwise Separable Conv):
     - EOG nhánh: DWSep kernel=51 → pool → embed
     - EEG nhánh: DWSep kernel=15 → pool → embed
     - Concat + Linear → embed_dim=64

  2. Mini GRU (hidden_size=32, num_layers=1):
     - Quét tuần tự qua 21 epochs
     - O(1) RAM khi chạy inference real-time
     - Lấy hidden state cuối → Classifier

  3. Adaptor Layer (CNN embed→256, GRU hidden→256):
     - Project features lên không gian 256-dim của Student V1 projected features
     - Dùng cho Feature Matching MSE Loss khi distill từ Student V1

Ước tính params: ~40K
Sau INT8 Quantize: ~10KB file
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────
# Depthwise Separable Conv 1D block
# ─────────────────────────────────────────────────────────────

class DWSepConv1d(nn.Module):
    """
    Depthwise Separable Convolution 1D.
    Thay thế Conv1d thường để giảm params 5–10x.
    Bước 1 (Depthwise): mỗi channel xử lý riêng biệt.
    Bước 2 (Pointwise): trộn thông tin các channel lại.
    """
    def __init__(self, in_ch: int, out_ch: int, kernel: int,
                 stride: int = 1, groups_dw: int = None):
        super().__init__()
        groups_dw = groups_dw or in_ch
        pad = kernel // 2
        self.dw = nn.Conv1d(in_ch, in_ch, kernel_size=kernel, stride=stride,
                            padding=pad, groups=groups_dw, bias=False)
        self.pw = nn.Conv1d(in_ch, out_ch, kernel_size=1, bias=False)
        num_groups = min(4, out_ch)
        while out_ch % num_groups != 0:
            num_groups //= 2
        self.norm = nn.GroupNorm(num_groups, out_ch)
        self.act  = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.norm(self.pw(self.dw(x))))


# ─────────────────────────────────────────────────────────────
# DWSep Multi-branch CNN (siêu nhẹ)
# ─────────────────────────────────────────────────────────────

class DWSepMultiBranchCNN(nn.Module):
    """
    2 nhánh Depthwise Separable Conv:
      - Nhánh EOG (1ch, kernel=51): bắt Slow Eye Movements
      - Nhánh EEG (2ch, kernel=15): bắt alpha dropout

    Input:  (B, 3, T=3000)
    Output: (B, embed_dim)
    """
    def __init__(self, eog_ch: int = 16, eeg_ch: int = 16,
                 embed_dim: int = 64, stride: int = 5, dropout: float = 0.1):
        super().__init__()
        # EOG branch: 1 → eog_ch
        self.eog_branch = nn.Sequential(
            DWSepConv1d(1, eog_ch, kernel=51, stride=stride),
            DWSepConv1d(eog_ch, eog_ch, kernel=5, stride=2),
        )
        # EEG branch: 2 → eeg_ch
        self.eeg_branch = nn.Sequential(
            DWSepConv1d(2, eeg_ch, kernel=15, stride=stride),
            DWSepConv1d(eeg_ch, eeg_ch, kernel=5, stride=2),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Sequential(
            nn.Linear(eog_ch + eeg_ch, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, T) → (B, embed_dim)"""
        eeg = self.eeg_branch(x[:, :2, :])   # (B, eeg_ch, T')
        eog = self.eog_branch(x[:, 2:3, :])  # (B, eog_ch, T')
        min_t = min(eeg.shape[-1], eog.shape[-1])
        merged = torch.cat([eeg[..., :min_t], eog[..., :min_t]], dim=1)
        merged = self.pool(merged).squeeze(-1)  # (B, eog_ch+eeg_ch)
        return self.proj(merged)                # (B, embed_dim)


# ─────────────────────────────────────────────────────────────
# Student V2: DWSep CNN + Mini GRU
# ─────────────────────────────────────────────────────────────

class StudentSleepNetV2(nn.Module):
    """
    Student V2 — Edge AI tối thượng.

    Args:
        embed_dim:      Chiều CNN embedding. Mặc định 64.
        gru_hidden:     Số neurons GRU. Mặc định 32.
        gru_layers:     Số layer GRU. Mặc định 1.
        v1_embed_dim:   Chiều Student V1 (Teacher của V2), để Adaptor. Mặc định 128.
        num_classes:    Số nhãn. Mặc định 5.
        seq_len:        Chiều dài chuỗi. Mặc định 21.
        dropout:        Dropout. Mặc định 0.1.
    """
    def __init__(self,
                 embed_dim:   int = 64,
                 gru_hidden:  int = 32,
                 gru_layers:  int = 1,
                 v1_embed_dim: int = 128,
                 num_classes: int = 5,
                 seq_len:     int = 21,
                 dropout:     float = 0.1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.gru_hidden  = gru_hidden
        self.seq_len     = seq_len
        self.v1_embed_dim = v1_embed_dim

        # Khối 1: DWSep CNN
        self.cnn = DWSepMultiBranchCNN(
            eog_ch=embed_dim // 4,
            eeg_ch=embed_dim // 4 * 3,  # EEG nhận nhiều filter hơn (2 kênh)
            embed_dim=embed_dim,
            stride=5,
            dropout=dropout,
        )

        # Khối 2: Mini GRU
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )
        self.gru_norm = nn.LayerNorm(gru_hidden)
        self.gru_dropout = nn.Dropout(dropout)

        # Adaptor: GRU hidden → Student V1 space (cho Feature Matching Loss)
        # cnn_feat_proj: embed_dim → v1_embed_dim
        self.cnn_adaptor = nn.Linear(embed_dim, v1_embed_dim)
        # ctx_feat_proj: gru_hidden → v1_embed_dim
        self.ctx_adaptor = nn.Linear(gru_hidden, v1_embed_dim)

        # Classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(gru_hidden),
            nn.Linear(gru_hidden, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> dict:
        """
        x: (B, L, 3, T)
        returns: {
            'logits'        : (B, num_classes),
            'cnn_feat_proj' : (B, v1_embed_dim)  → khớp với student_v1['cnn_feat'],
            'ctx_feat_proj' : (B, v1_embed_dim)  → khớp với student_v1['ctx_feat']
        }
        """
        B, L, C, T = x.shape

        # ① CNN Feature extraction — xử lý song song toàn bộ sequence
        x_flat = x.reshape(B * L, C, T)        # (B*L, 3, T)
        feats  = self.cnn(x_flat)               # (B*L, embed_dim)
        feats  = feats.view(B, L, self.embed_dim)  # (B, L, embed_dim)

        # ② CNN feat của epoch trung tâm (để Feature Matching)
        center   = L // 2
        cnn_feat = feats[:, center, :]           # (B, embed_dim)

        # ③ GRU quét qua chuỗi sequence
        gru_out, hn = self.gru(feats)            # gru_out: (B, L, gru_hidden)
        # Lấy hidden state của epoch trung tâm
        ctx_feat = self.gru_norm(
            self.gru_dropout(gru_out[:, center, :])
        )                                        # (B, gru_hidden)

        # ④ Adaptor projection → Student V1 space
        cnn_feat_proj = self.cnn_adaptor(cnn_feat)   # (B, v1_embed_dim)
        ctx_feat_proj = self.ctx_adaptor(ctx_feat)   # (B, v1_embed_dim)

        # ⑤ Classifier
        logits = self.classifier(ctx_feat)           # (B, num_classes)

        return {
            'logits'        : logits,
            'cnn_feat'      : cnn_feat,       # native features (used when this model is a KD source)
            'ctx_feat'      : ctx_feat,
            'cnn_feat_proj' : cnn_feat_proj,
            'ctx_feat_proj' : ctx_feat_proj,
        }

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


if __name__ == '__main__':
    model = StudentSleepNetV2()
    x = torch.randn(4, 21, 3, 3000)
    out = model(x)
    print(f'StudentSleepNetV2 | Params: {model.count_parameters():,}')
    print(f'  logits:          {out["logits"].shape}')
    print(f'  cnn_feat_proj:   {out["cnn_feat_proj"].shape}')
    print(f'  ctx_feat_proj:   {out["ctx_feat_proj"].shape}')

In [ ]:
%%writefile /content/skd_code/losses_v11.py
# -*- coding: utf-8 -*-
"""
losses_v11.py
Loss functions cho V11 Teacher-Student Knowledge Distillation.

Gồm:
  1. FocalLoss            : Cho Teacher — phạt nặng khi sai N1/Wake
  2. KDLossV11            : Knowledge Distillation đầy đủ 3 thành phần:
                            - Hard Label Loss (CE)
                            - Soft Label Loss (KL Divergence, Temperature T)
                            - Feature Matching Loss (MSE qua Adaptor Layer)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────
# 1. Focal Loss — Teacher Training
# ─────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    Focal Loss cho phân loại đa lớp mất cân bằng.
    Từ "Focal Loss for Dense Object Detection" (Lin et al., 2017).

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Args:
        gamma:         Hệ số tập trung (focusing param). Cao hơn → phạt mạnh hơn mẫu dễ.
                       Mặc định 2.0 (giá trị gốc của paper).
        class_weights: Trọng số từng lớp (Tensor shape [num_classes]).
                       Dùng để phạt nặng hơn lớp N1 (index 1).
                       Nếu None, sẽ tự tính class weight từ tần suất.
        label_smoothing: Làm mịn nhãn để tránh overfit. Mặc định 0.05.
    """
    def __init__(self, gamma: float = 2.0,
                 class_weights: torch.Tensor = None,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.gamma           = gamma
        self.class_weights   = class_weights  # Sẽ được đưa lên device khi dùng
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits:  (B, num_classes) — raw logits
            targets: (B,)             — ground truth labels (integer)
        Returns:
            scalar loss
        """
        # Tính log_softmax + softmax
        log_prob = F.log_softmax(logits, dim=-1)    # (B, num_classes)
        prob     = log_prob.exp()                    # (B, num_classes)

        # Lấy xác suất đúng nhãn cho mỗi mẫu
        # p_t = prob của nhãn đúng
        p_t      = prob.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)

        # Focal weight = (1 - p_t)^gamma
        focal_w  = (1.0 - p_t) ** self.gamma                        # (B,)

        # Tính cross-entropy per-sample (có label smoothing)
        num_cls  = logits.size(1)
        smooth   = self.label_smoothing
        # CE với label smoothing: CE = (1-smooth)*CE_hard + smooth*CE_uniform
        ce_hard  = -log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)  # (B,)
        ce_unif  = -log_prob.mean(dim=-1)                                 # (B,)
        ce       = (1 - smooth) * ce_hard + smooth * ce_unif             # (B,)

        # Áp class weight nếu có
        if self.class_weights is not None:
            w = self.class_weights.to(logits.device)[targets]             # (B,)
            ce = ce * w

        loss = focal_w * ce
        return loss.mean()


# ─────────────────────────────────────────────────────────────
# 2. KDLossV11 — Knowledge Distillation đầy đủ
# ─────────────────────────────────────────────────────────────

class KDLossV11(nn.Module):
    """
    Knowledge Distillation Loss V11 — kết hợp 3 thành phần:

      L_total = w_ce  * L_CE(student_logits, hard_labels)       [Hard Label]
              + w_kl  * T² * KL(student_soft || teacher_soft)   [Soft Label]
              + w_mse * (MSE(cnn_proj, teacher_cnn)             [Feature Match]
                        + MSE(ctx_proj, teacher_ctx))

    Với Adaptor Layer đã được tích hợp sẵn trong Student Model,
    MSE được tính giữa:
      - student.cnn_feat_proj ↔ teacher.cnn_feat
      - student.ctx_feat_proj ↔ teacher.ctx_feat

    Args:
        temperature: Nhiệt độ làm mịn Soft Labels. Nên cao (4-8) khi dùng Focal Loss.
        w_ce:        Trọng số Hard Label Loss. Mặc định 0.3.
        w_kl:        Trọng số Soft KL Loss.   Mặc định 0.5.
        w_mse:       Trọng số Feature Match.  Mặc định 0.2 (tùy chọn, có thể tắt).
        label_smoothing: Label smoothing cho CE. Mặc định 0.05.
    """
    def __init__(self,
                 temperature: float = 5.0,
                 w_ce:        float = 0.3,
                 w_kl:        float = 0.5,
                 w_mse:       float = 0.2,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.T      = temperature
        self.w_ce   = w_ce
        self.w_kl   = w_kl
        self.w_mse  = w_mse
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self,
                student_out:  dict,
                teacher_out:  dict,
                targets:      torch.Tensor) -> dict:
        """
        Args:
            student_out : dict từ StudentSleepNetV11.forward()
                          {logits, cnn_feat_proj, ctx_feat_proj}
            teacher_out : dict từ TeacherSleepNetV11.forward() — DETACHED
                          {logits, cnn_feat, ctx_feat}
            targets     : (B,) ground truth labels
        Returns:
            dict: {loss, ce_loss, kl_loss, mse_loss}
        """
        s_logits = student_out['logits']           # (B, C)
        t_logits = teacher_out['logits'].detach()  # Detach — không train Teacher
        t_cnn    = teacher_out['cnn_feat'].detach()
        t_ctx    = teacher_out['ctx_feat'].detach()

        # ── 1. Hard Label Loss ─────────────────────────────────
        ce_loss = self.ce_loss(s_logits, targets)

        # ── 2. Soft Label KL Loss với Temperature ─────────────
        # T cao → làm phẳng phân phối sharp của Teacher (do Focal Loss gây ra)
        s_soft = F.log_softmax(s_logits / self.T, dim=-1)
        t_soft = F.softmax(t_logits / self.T, dim=-1)
        kl_loss = F.kl_div(s_soft, t_soft, reduction='batchmean') * (self.T ** 2)

        # ── 3. Feature Matching MSE Loss (qua Adaptor Layer) ──
        # student cnn_feat_proj đã được project về teacher_embed_dim
        mse_cnn = F.mse_loss(student_out['cnn_feat_proj'], t_cnn)
        mse_ctx = F.mse_loss(student_out['ctx_feat_proj'], t_ctx)
        mse_loss = (mse_cnn + mse_ctx) / 2.0

        # ── Tổng hợp ──────────────────────────────────────────
        total = (self.w_ce  * ce_loss +
                 self.w_kl  * kl_loss +
                 self.w_mse * mse_loss)

        return {
            'loss'    : total,
            'ce_loss' : ce_loss.item(),
            'kl_loss' : kl_loss.item(),
            'mse_loss': mse_loss.item(),
        }

In [ ]:
%%writefile /content/skd_code/prep.py
# prep.py — download + preprocess Sleep-EDF Expanded (v1.0.0) and HMC (v1.1) per recording; subject-level folds.
import os, re, glob, random, time
import numpy as np
import pandas as pd
import scipy.signal
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

PHYSIONET = 'https://physionet.org/files/sleep-edfx/1.0.0'
S3_BUCKET = 'https://physionet-open.s3.amazonaws.com'          # official AWS Open Data mirror (much faster)
S3_PREFIX = 'sleep-edfx/1.0.0'
SUBSETS = ['sleep-cassette', 'sleep-telemetry']
CHANNELS = ['EEG Fpz-Cz', 'EEG Pz-Oz', 'EOG horizontal']   # slot 0, 1 = EEG; slot 2 = EOG
FS, EPOCH_S = 100, 30
EP = FS * EPOCH_S
STAGE_MAP = {'Sleep stage W': 0, 'Sleep stage 1': 1, 'Sleep stage 2': 2,
             'Sleep stage 3': 3, 'Sleep stage 4': 3, 'Sleep stage R': 4}   # M / ? -> -1
# Same filter as the original preprocess.py: Butterworth order 5, 0.5-35 Hz, zero-phase, per epoch
B_BP, A_BP = scipy.signal.butter(N=5, Wn=[0.5, 35.0], btype='bandpass', fs=FS)

# HMC (Haaglanden Medisch Centrum sleep staging database), same channel mapping as the original extract_hmc_to_numpy.py
HMC_FS = 256
HMC_CHANNELS = {'F4': ['EEG F4-M1', 'EEG F4-A1'], 'O2': ['EEG O2-M1', 'EEG O2-A1'],
                'E1': ['EOG E1-M2', 'EOG E1-A2'], 'E2': ['EOG E2-M2', 'EOG E2-A2']}   # slots: F4-M1, O2-M1, E1-E2
HMC_STAGES = {'Sleep stage W': 0, 'Sleep stage N1': 1, 'Sleep stage N2': 2, 'Sleep stage N3': 3, 'Sleep stage R': 4}


# ---------------------------------------------------------------- download
def _list_edf(url):
    html = requests.get(url, timeout=120).text
    return sorted(set(re.findall(r'href="([^"/?]+\.edf)"', html)))


def _fetch(url, path, retries=4):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return 'skip'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=300) as r:
                r.raise_for_status()
                tmp = path + '.part'
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            os.replace(tmp, path)
            return 'ok'
        except Exception as e:
            if attempt == retries - 1:
                return f'FAIL {e}'
            time.sleep(5 * (attempt + 1))


def _s3_keys(prefix):
    keys, token = [], None
    while True:
        params = {'list-type': 2, 'prefix': prefix}
        if token:
            params['continuation-token'] = token
        xml = requests.get(S3_BUCKET + '/', params=params, timeout=120).text
        keys += re.findall(r'<Key>([^<]+)</Key>', xml)
        m = re.search(r'<NextContinuationToken>([^<]+)</NextContinuationToken>', xml)
        if not m:
            return keys
        token = m.group(1)


def _list_s3(sub):
    return [k.rsplit('/', 1)[1] for k in _s3_keys(f'{S3_PREFIX}/{sub}/') if k.endswith('.edf')]


def download_sleep_edfx(raw_dir, n_threads=8, source='s3'):
    jobs = []
    for sub in SUBSETS:
        if source == 's3':
            base, names = f'{S3_BUCKET}/{S3_PREFIX}/{sub}/', _list_s3(sub)
        else:
            base, names = f'{PHYSIONET}/{sub}/', _list_edf(f'{PHYSIONET}/{sub}/')
        jobs += [(base + f, os.path.join(raw_dir, sub, f)) for f in names]
    print(f'{len(jobs)} EDF files listed ({source})')
    status = {}
    with ThreadPoolExecutor(n_threads) as ex:
        futs = {ex.submit(_fetch, u, p): p for u, p in jobs}
        for k, fu in enumerate(as_completed(futs), 1):
            status[futs[fu]] = fu.result()
            if k % 50 == 0:
                print(f'  {k}/{len(jobs)}')
    fails = {p: s for p, s in status.items() if s.startswith('FAIL')}
    print(f'done | failed: {len(fails)}')
    for p, s in fails.items():
        print('  ', p, s)
    return fails


def pair_recordings(raw_dir):
    psgs = sorted(glob.glob(os.path.join(raw_dir, '**', '*-PSG.edf'), recursive=True))
    hyps = {os.path.basename(h)[:7]: h
            for h in glob.glob(os.path.join(raw_dir, '**', '*-Hypnogram.edf'), recursive=True)}
    pairs = []
    for p in psgs:
        rec = os.path.basename(p).split('-')[0]            # e.g. SC4001E0
        h = hyps.get(rec[:7])
        if h is None:
            print('no hypnogram for', rec)
            continue
        pairs.append((rec, p, h))
    return pairs


# ---------------------------------------------------------------- shared epoch processing
def subject_of(rec):
    return rec[:5]                                          # SC4ss / ST7ss


def _finalize(rec, x, y, out_dir, trim_wake_min, store_full_night, reject_ptp_uv, pad):
    """x: (n,3,3000) raw µV @100 Hz, y: (n,) int8 (-1 = not a target). Filter, z-score, trim, save.
    Returns the per-recording summary fields shared by all datasets."""
    n = len(y)
    ptp = np.ptp(x, axis=2).max(axis=1).astype(np.float32)  # raw µV, max over channels (reported, optional exclusion)
    x = scipy.signal.filtfilt(B_BP, A_BP, x, axis=2)
    x = (x - x.mean(axis=2, keepdims=True)) / (x.std(axis=2, keepdims=True) + 1e-6)

    labelled = y >= 0
    if reject_ptp_uv is not None:
        labelled &= ptp <= reject_ptp_uv
    sleep = np.where((y >= 1) & (y <= 4))[0]
    if len(sleep) == 0:
        raise ValueError(f'{rec}: no sleep epochs')
    if trim_wake_min is None:
        ts, te = 0, n
    else:
        m = int(trim_wake_min * 60 / EPOCH_S)
        ts, te = max(0, sleep[0] - m), min(n, sleep[-1] + m + 1)
    centers_full = np.where(labelled)[0]
    centers_trim = centers_full[(centers_full >= ts) & (centers_full < te)]

    if store_full_night:
        lo, hi = 0, n
    else:                                                    # keep only the trimmed span + context margin
        lo, hi = max(0, ts - pad), min(n, te + pad)
        centers_trim = centers_trim - lo
        centers_full = np.zeros(0, dtype=np.int64)
    np.savez(os.path.join(out_dir, f'{rec}.npz'), x=x[lo:hi].astype(np.float16), y=y[lo:hi], ptp=ptp[lo:hi],
             centers_trim=centers_trim.astype(np.int64), centers_full=centers_full.astype(np.int64),
             lo=lo, n_total=n)

    yt = y[ts:te][labelled[ts:te]]
    yf = y[labelled]
    row = dict(n_epochs_total=n, trim_start=ts, trim_end=te, n_stored=hi - lo, n_unlabelled=int((y < 0).sum()),
               n_ptp_over_150uv=int((ptp > 150).sum()))
    for c, nm in enumerate(['W', 'N1', 'N2', 'N3', 'REM']):
        row[f'trim_{nm}'] = int((yt == c).sum())
        row[f'full_{nm}'] = int((yf == c).sum())
    return row


# ---------------------------------------------------------------- Sleep-EDF
def process_recording(rec, psg, hyp, out_dir, trim_wake_min=30, store_full_night=False,
                      reject_ptp_uv=None, pad=10):
    import mne
    raw = mne.io.read_raw_edf(psg, preload=False, verbose='ERROR')
    missing = [c for c in CHANNELS if c not in raw.ch_names]
    if missing:
        raise ValueError(f'{rec}: missing channels {missing}')
    raw.pick(CHANNELS)
    raw.reorder_channels(CHANNELS)
    raw.load_data(verbose='ERROR')
    sfreq_orig = float(raw.info['sfreq'])
    if sfreq_orig != FS:
        raw.resample(FS, verbose='ERROR')
    raw.set_annotations(mne.read_annotations(hyp), emit_warning=False)

    data = raw.get_data() * 1e6                             # volts -> microvolts, (3, N)
    n = data.shape[1] // EP
    x = data[:, :n * EP].reshape(3, n, EP).transpose(1, 0, 2).astype(np.float32)
    y = np.full(n, -1, dtype=np.int8)
    for on, du, de in zip(raw.annotations.onset, raw.annotations.duration, raw.annotations.description):
        s = int(round((on - raw.first_time) / EPOCH_S))
        e = s + int(round(du / EPOCH_S))
        y[max(s, 0):min(e, n)] = STAGE_MAP.get(de, -1)

    row = dict(rec=rec, subject=subject_of(rec), cohort=rec[:2], night=int(rec[5]), sfreq_orig=sfreq_orig)
    row.update(_finalize(rec, x, y, out_dir, trim_wake_min, store_full_night, reject_ptp_uv, pad))
    return row


def preprocess_all(raw_dir, out_dir, meta_csv, **kw):
    os.makedirs(out_dir, exist_ok=True)
    old = pd.read_csv(meta_csv) if os.path.exists(meta_csv) else pd.DataFrame()
    done = set(old['rec']) if len(old) else set()
    rows = old.to_dict('records') if len(old) else []
    pairs = pair_recordings(raw_dir)
    print(f'{len(pairs)} recordings paired | already processed: {len(done)}')
    for k, (rec, psg, hyp) in enumerate(pairs, 1):
        if rec in done and os.path.exists(os.path.join(out_dir, f'{rec}.npz')):
            continue
        t0 = time.time()
        rows.append(process_recording(rec, psg, hyp, out_dir, **kw))
        pd.DataFrame(rows).to_csv(meta_csv, index=False)     # checkpoint progress
        print(f'  [{k}/{len(pairs)}] {rec} ({time.time() - t0:.1f}s)')
    meta = pd.DataFrame(rows).sort_values('rec').reset_index(drop=True)
    meta.to_csv(meta_csv, index=False)
    return meta


# ---------------------------------------------------------------- HMC
def parse_hmc_scoring(txt_path):
    """(onset_s, label) for every 30-s 'Sleep stage *' row of SNxxx_sleepscoring.txt (lights on/off etc. ignored)."""
    out = []
    for line in open(txt_path, encoding='utf-8', errors='ignore').read().splitlines()[1:]:
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 5 and parts[4] in HMC_STAGES and abs(float(parts[3]) - EPOCH_S) < 0.1:
            out.append((float(parts[2]), HMC_STAGES[parts[4]]))
    return sorted(out)


def process_hmc_recording(rec, edf, txt, out_dir, trim_wake_min=30, store_full_night=False,
                          reject_ptp_uv=None, pad=10):
    import mne
    raw = mne.io.read_raw_edf(edf, preload=False, verbose='ERROR')
    names = {c.strip().lower(): c for c in raw.ch_names}
    pick = {}
    for slot, cands in HMC_CHANNELS.items():
        hit = [names[c.lower()] for c in cands if c.lower() in names]
        if not hit:
            raise ValueError(f'{rec}: missing {slot} ({cands}); available {raw.ch_names}')
        pick[slot] = hit[0]
    sfreq_orig = float(raw.info['sfreq'])
    if abs(sfreq_orig - HMC_FS) > 0.5:
        raise ValueError(f'{rec}: sfreq {sfreq_orig}')
    raw.pick([pick[s] for s in ('F4', 'O2', 'E1', 'E2')])
    d = raw.get_data(picks=[pick[s] for s in ('F4', 'O2', 'E1', 'E2')]) * 1e6
    sig = np.stack([d[0], d[1], d[2] - d[3]])               # F4-M1, O2-M1, E1-E2 (derived horizontal EOG)

    stages = parse_hmc_scoring(txt)
    ep256 = HMC_FS * EPOCH_S
    segs, labs, onsets = [], [], []
    for on, lab in stages:
        s = int(round(on * HMC_FS))
        if s >= 0 and s + ep256 <= sig.shape[1]:
            segs.append(sig[:, s:s + ep256]); labs.append(lab); onsets.append(on)
    x = scipy.signal.resample_poly(np.stack(segs), up=25, down=64, axis=2).astype(np.float32)   # 256 -> 100 Hz
    assert x.shape[2] == EP
    y = np.array(labs, dtype=np.int8)
    n_gaps = int((np.diff(onsets) != EPOCH_S).sum())         # non-contiguous scored epochs (context would jump)

    row = dict(rec=rec, subject=rec, cohort='HMC', night=1, sfreq_orig=sfreq_orig, n_gaps=n_gaps,
               channels='|'.join(pick[s] for s in ('F4', 'O2', 'E1', 'E2')))
    row.update(_finalize(rec, x, y, out_dir, trim_wake_min, store_full_night, reject_ptp_uv, pad))
    return row


def list_hmc(version='1.1'):
    """Recording ids (SNxxx) that have both the PSG EDF and the scoring txt on the S3 mirror."""
    keys = _s3_keys(f'hmc-sleep-staging/{version}/recordings/')
    edf = {m.group(1) for k in keys for m in [re.search(r'/(SN\d+)\.edf$', k)] if m}
    txt = {m.group(1) for k in keys for m in [re.search(r'/(SN\d+)_sleepscoring\.txt$', k)] if m}
    recs = sorted(edf & txt)
    if not recs:
        raise RuntimeError(f'no HMC recordings found for version {version} ({len(keys)} keys listed)')
    return recs


def download_process_hmc(raw_dir, out_dir, meta_csv, version='1.1', n_threads=4, keep_raw=False, **kw):
    """Download each recording from the S3 mirror, process it, delete the raw EDF (unless keep_raw). Resumable."""
    os.makedirs(out_dir, exist_ok=True)
    prefix = f'hmc-sleep-staging/{version}/recordings/'
    recs = list_hmc(version)
    old = pd.read_csv(meta_csv) if os.path.exists(meta_csv) else pd.DataFrame()
    rows = old.to_dict('records') if len(old) else []
    done = {r['rec'] for r in rows if os.path.exists(os.path.join(out_dir, f"{r['rec']}.npz"))}
    todo = [r for r in recs if r not in done]
    print(f'HMC v{version}: {len(recs)} recordings | already processed: {len(done)} | to do: {len(todo)}')

    def get(rec):
        for f in (f'{rec}.edf', f'{rec}_sleepscoring.txt'):
            st = _fetch(f'{S3_BUCKET}/{prefix}{f}', os.path.join(raw_dir, f))
            if st.startswith('FAIL'):
                return st
        return 'ok'

    with ThreadPoolExecutor(n_threads) as ex:                # downloads run ahead of processing
        futs = [(rec, ex.submit(get, rec)) for rec in todo]
        for k, (rec, fu) in enumerate(futs, 1):
            st = fu.result()
            if st != 'ok':
                print(f'  [{k}/{len(todo)}] {rec}: download {st}')
                continue
            t0 = time.time()
            edf, txt = os.path.join(raw_dir, f'{rec}.edf'), os.path.join(raw_dir, f'{rec}_sleepscoring.txt')
            rows = [r for r in rows if r['rec'] != rec] + [process_hmc_recording(rec, edf, txt, out_dir, **kw)]
            pd.DataFrame(rows).to_csv(meta_csv, index=False)
            if not keep_raw:
                os.remove(edf)
            print(f'  [{k}/{len(todo)}] {rec} ({time.time() - t0:.1f}s)')
    meta = pd.DataFrame(rows).sort_values('rec').reset_index(drop=True)
    meta.to_csv(meta_csv, index=False)
    return meta


# ---------------------------------------------------------------- subject-level folds
def make_subject_folds(meta, k=5, seed=42):
    """Shuffle subjects (seed) within each cohort, assign round-robin -> every night of a subject in one fold.
    Cohorts are processed in sorted order (Sleep-EDF: SC then ST; HMC: one cohort) with a single RNG."""
    rng = random.Random(seed)
    fold_of = {}
    for cohort in sorted(meta.cohort.unique()):
        subs = sorted(meta.loc[meta.cohort == cohort, 'subject'].unique())
        rng.shuffle(subs)
        for j, s in enumerate(subs):
            fold_of[s] = j % k
    meta = meta.copy()
    meta['fold'] = meta['subject'].map(fold_of)
    assert meta.groupby('subject')['fold'].nunique().max() == 1, 'subject spans >1 fold'
    folds = {f: sorted(meta.loc[meta.fold == f, 'rec']) for f in range(k)}
    return meta, folds


def fold_role_lists(meta, k=5):
    """Explicit train / validation / test subjects and recordings for every CV fold (test = i, val = i+1 mod k)."""
    out = {}
    for i in range(k):
        roles = {'test': [i], 'validation': [(i + 1) % k], 'train': [j for j in range(k) if j not in (i, (i + 1) % k)]}
        out[f'fold{i}'] = {r: dict(source_folds=fs,
                                   subjects=sorted(meta.loc[meta.fold.isin(fs), 'subject'].unique().tolist()),
                                   recordings=sorted(meta.loc[meta.fold.isin(fs), 'rec'].tolist()))
                           for r, fs in roles.items()}
    return out

In [ ]:
%%writefile /content/skd_code/skd.py
# skd.py — training / evaluation / statistics for the subject-level re-run.
import os, json, time, random, shutil, contextlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix

from model_v11.teacher_v11 import TeacherSleepNetV11
from model_v11.student_v11 import StudentSleepNetV11
from model_v11.student_v2 import StudentSleepNetV2
from losses_v11 import FocalLoss, KDLossV11

SEQ_LEN = 21
STAGES = ['W', 'N1', 'N2', 'N3', 'REM']
NATIVE_DIM = {'teacher': 256, 'assistant': 128}   # feature size a source model exposes for MSE matching


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(device, use_bf16):
    if device.type == 'cuda' and use_bf16 and torch.cuda.is_bf16_supported():
        return torch.autocast('cuda', dtype=torch.bfloat16)
    return contextlib.nullcontext()


# ================================================================ data
class Store:
    """All recordings in RAM: x (n,3,3000) fp16, y (n,), centers = epoch indices used as targets."""
    def __init__(self, data_dir, recs):
        self.recs, self.pos = [], {}
        for rid in recs:
            d = np.load(os.path.join(data_dir, f'{rid}.npz'))
            self.pos[rid] = len(self.recs)
            self.recs.append(dict(id=rid, x=d['x'], y=d['y'].astype(np.int64),
                                  trim=d['centers_trim'], full=d['centers_full']))

    def has_full(self):
        return all(len(r['full']) for r in self.recs)


class SeqDS(Dataset):
    """21-epoch windows centred on each target epoch; context never crosses a recording boundary
    (indices are clamped to the recording, i.e. edge-replicate padding)."""
    def __init__(self, store, recs, which='trim', seq_len=SEQ_LEN):
        self.store, self.half = store, seq_len // 2
        rows = [np.stack([np.full(len(store.recs[store.pos[r]][which]), store.pos[r]),
                          store.recs[store.pos[r]][which]], 1) for r in recs]
        self.index = np.concatenate(rows).astype(np.int64)

    def __len__(self):
        return len(self.index)

    def labels(self):
        return np.array([self.store.recs[r]['y'][c] for r, c in self.index])

    def rec_ids(self):
        return np.array([self.store.recs[r]['id'] for r, _ in self.index])

    def __getitem__(self, i):
        ri, c = self.index[i]
        r = self.store.recs[ri]
        idx = np.clip(np.arange(c - self.half, c + self.half + 1), 0, len(r['y']) - 1)
        return torch.from_numpy(r['x'][idx].astype(np.float32)), int(r['y'][c]), i


def split_recs(folds, i, k=5):
    test, val = folds[i], folds[(i + 1) % k]
    train = [r for j in range(k) if j not in (i, (i + 1) % k) for r in folds[j]]
    return train, val, test


# ================================================================ models
def build_model(kind, target_dim=None):
    if kind == 'teacher':
        return TeacherSleepNetV11(embed_dim=256, num_heads=8, num_layers=3, num_classes=5,
                                  seq_len=SEQ_LEN, dropout=0.1)
    if kind == 'assistant':
        return StudentSleepNetV11(embed_dim=128, teacher_embed_dim=target_dim or 256, num_heads=4,
                                  num_layers=2, num_classes=5, seq_len=SEQ_LEN, dropout=0.1)
    if kind == 'edge':
        return StudentSleepNetV2(embed_dim=64, gru_hidden=32, gru_layers=1, v1_embed_dim=target_dim or 128,
                                 num_classes=5, seq_len=SEQ_LEN, dropout=0.1)
    raise ValueError(kind)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    adaptor = sum(p.numel() for n, p in model.named_parameters() if 'adaptor' in n)
    return total, total - adaptor


# ================================================================ metrics
def metrics(y, p):
    f1 = f1_score(y, p, labels=list(range(5)), average=None, zero_division=0)
    return dict(acc=float((y == p).mean()), mf1=float(f1.mean()), kappa=float(cohen_kappa_score(y, p)),
                f1=[float(v) for v in f1], n=int(len(y)),
                cm=confusion_matrix(y, p, labels=list(range(5))).tolist())


@torch.no_grad()
def predict(model, ds, C, device):
    model.eval()
    out = []
    for x, _, _ in DataLoader(ds, batch_size=C['EVAL_BS'], shuffle=False,
                              num_workers=C['NUM_WORKERS'], pin_memory=True):
        with amp_ctx(device, C['USE_BF16']):
            out.append(model(x.to(device, non_blocking=True))['logits'].float().cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def cache_source(src, ds, C, device):
    """Frozen source outputs for every training window (eval mode, no augmentation -> identical to on-the-fly)."""
    src.eval()
    cache = None
    for x, _, i in DataLoader(ds, batch_size=C['EVAL_BS'], shuffle=False,
                              num_workers=C['NUM_WORKERS'], pin_memory=True):
        with amp_ctx(device, C['USE_BF16']):
            o = src(x.to(device, non_blocking=True))
        if cache is None:
            cache = {'logits': torch.empty(len(ds), 5, device=device),
                     'cnn_feat': torch.empty(len(ds), o['cnn_feat'].shape[1], dtype=torch.float16, device=device),
                     'ctx_feat': torch.empty(len(ds), o['ctx_feat'].shape[1], dtype=torch.float16, device=device)}
        i = i.to(device)
        cache['logits'][i] = o['logits'].float()
        cache['cnn_feat'][i] = o['cnn_feat'].half()
        cache['ctx_feat'][i] = o['ctx_feat'].half()
    return cache


# ================================================================ runs
def run_dir(C, arm_name, fold, seed):
    return os.path.join(C['RUNS_DIR'], f'seed{seed}', arm_name, f'fold{fold}')


def load_result(C, arm_name, fold, seed):
    p = os.path.join(run_dir(C, arm_name, fold, seed), 'result.json')
    return json.load(open(p)) if os.path.exists(p) else None


def resolve_source(C, src, fold, seed):
    return src


def resolve_tuned(C, arm):
    """Arm with arm['tune'] = dict(prefix=..., fold=f): take lr / label smoothing of the grid arm (names starting
    with prefix, trained on fold f only, first seed) that has the best VALIDATION Macro-F1. Test data never used."""
    t = arm['tune']
    seed0 = C['SEEDS'][0]
    grid = [a for a in C['ARMS'] if a['name'].startswith(t['prefix'])]
    res = {a['name']: load_result(C, a['name'], t['fold'], seed0) for a in grid}
    missing = [n for n, r in res.items() if r is None]
    if missing:
        raise RuntimeError(f'tuning grid not finished (fold {t["fold"]}, seed {seed0}): {missing}')
    best = max(grid, key=lambda a: res[a['name']]['best_val_mf1'])   # ties -> earliest grid entry (list order)
    out = {k: v for k, v in arm.items() if k != 'tune'}
    out.update(lr=best.get('lr', C['LR']), ls=best.get('ls', C['LABEL_SMOOTHING']), selected_from=best['name'],
               tuning={n: r['best_val_mf1'] for n, r in res.items()})
    return out


def _reuse_tuning_run(C, arm, fold, seed):
    """The selected grid run already is this arm's (fold, seed) run with identical settings: copy it, don't retrain."""
    src, dst = run_dir(C, arm['selected_from'], fold, seed), run_dir(C, arm['name'], fold, seed)
    os.makedirs(dst, exist_ok=True)
    for f in ('best.pt', 'preds_test.npz'):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    r = load_result(C, arm['selected_from'], fold, seed)
    r.update(arm=arm['name'], selected_from=arm['selected_from'], tuning=arm['tuning'], reused_tuning_run=True)
    json.dump(r, open(os.path.join(dst, 'result.json'), 'w'), indent=1)
    print(f'  {arm["name"]} fold{fold} seed{seed}: reused tuning run {arm["selected_from"]}')
    return r


def run_all(C, store, folds, device, run_folds=None, run_arms=None):
    """Arms in list order (= dependency order). arm['folds'] / arm['seeds'] restrict where an arm runs
    (tuning-grid arms: folds=[0], seeds='first')."""
    for arm in C['ARMS']:
        if run_arms and arm['name'] not in run_arms:
            continue
        seeds = C['SEEDS'][:1] if arm.get('seeds') == 'first' else C['SEEDS']
        for seed in seeds:
            for fold in (run_folds if run_folds is not None else range(C['K_FOLDS'])):
                if 'folds' in arm and fold not in arm['folds']:
                    continue
                run_arm(arm, fold, seed, C, store, folds, device)


def run_arm(arm, fold, seed, C, store, folds, device):
    name = arm['name']
    odir = run_dir(C, name, fold, seed)
    if os.path.exists(os.path.join(odir, 'result.json')):
        print(f'[skip] {name} fold{fold} seed{seed} (done)')
        return load_result(C, name, fold, seed)
    if 'tune' in arm:
        tune_fold = arm['tune']['fold']
        arm = resolve_tuned(C, arm)
        print(f'  {name}: selected {arm["selected_from"]} (lr {arm["lr"]}, ls {arm["ls"]}) by validation Macro-F1 '
              f'on fold {tune_fold}')
        if fold == tune_fold and seed == C['SEEDS'][0]:
            return _reuse_tuning_run(C, arm, fold, seed)
    os.makedirs(odir, exist_ok=True)
    t_start = time.time()
    tr, va, te = split_recs(folds, fold, C['K_FOLDS'])
    train_ds, val_ds, test_ds = SeqDS(store, tr), SeqDS(store, va), SeqDS(store, te)
    y_val, y_test = val_ds.labels(), test_ds.labels()
    set_seed(seed * 1000 + fold)

    # ---- source (for KD arms)
    cache, src_name, target_dim = None, None, None
    src_seed = arm.get('src_seed', seed)        # fixed source seed: only the student's training seed varies
    if arm['loss'] == 'kd':
        src_name = resolve_source(C, arm['src'], fold, seed)
        sres = load_result(C, src_name, fold, src_seed)
        if sres is None:
            raise RuntimeError(f'source {src_name} fold{fold} seed{src_seed} not trained yet')
        assert sres['fold'] == fold and sres['seed'] == src_seed, 'source model from another fold / seed'
        src = build_model(sres['kind'], sres['target_dim']).to(device)
        src.load_state_dict(torch.load(os.path.join(run_dir(C, src_name, fold, src_seed), 'best.pt'),
                                       map_location=device, weights_only=True))
        target_dim = NATIVE_DIM[sres['kind']]
        t0 = time.time()
        cache = cache_source(src, train_ds, C, device)
        print(f'  cached {src_name} outputs for {len(train_ds):,} windows ({time.time() - t0:.0f}s)')
        del src
        torch.cuda.empty_cache()

    model = build_model(arm['kind'], target_dim).to(device)
    p_total, p_infer = count_params(model)

    # ---- loss
    ls = arm.get('ls', C['LABEL_SMOOTHING'])
    if arm['loss'] == 'focal':
        y_tr = train_ds.labels()
        counts = np.bincount(y_tr, minlength=5).astype(np.float64)
        cw = torch.tensor(counts.sum() / (5.0 * counts + 1e-8), dtype=torch.float32, device=device)
        crit = FocalLoss(gamma=C['FOCAL_GAMMA'], class_weights=cw, label_smoothing=ls)
    elif arm['loss'] == 'ce':
        crit = nn.CrossEntropyLoss(label_smoothing=ls)
    else:
        crit = KDLossV11(temperature=arm['T'], w_ce=C['W_CE'], w_kl=C['W_KL'], w_mse=C['W_MSE'],
                         label_smoothing=ls)

    lr = arm.get('lr', C['LR'])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=C['WD'])
    sch = CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2, eta_min=1e-6)
    st = dict(epoch=0, best=-1.0, best_epoch=0, patience=0, history=[])
    last_p, best_p = os.path.join(odir, 'last.pt'), os.path.join(odir, 'best.pt')
    if os.path.exists(last_p):                                # resume after a Colab disconnect
        ck = torch.load(last_p, map_location=device, weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch'])
        st = ck['state']
        print(f'  resumed at epoch {st["epoch"]}')

    g = torch.Generator().manual_seed(seed * 1000 + fold + st['epoch'])
    loader = DataLoader(train_ds, batch_size=arm['bs'], shuffle=True, drop_last=True, generator=g,
                        num_workers=C['NUM_WORKERS'], pin_memory=True, persistent_workers=C['NUM_WORKERS'] > 0)
    print(f'  {name} fold{fold} seed{seed} | params {p_total:,} (inference {p_infer:,}) | '
          f'train {len(train_ds):,} val {len(val_ds):,} test {len(test_ds):,} | lr {lr} ls {ls} bs {arm["bs"]}')

    for ep in range(st['epoch'] + 1, C['EPOCHS'] + 1):
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        for x, y, i in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(device, C['USE_BF16']):
                o = model(x)
            o = {k: v.float() for k, v in o.items()}
            if cache is not None:
                i = i.to(device)
                tgt = {'logits': cache['logits'][i], 'cnn_feat': cache['cnn_feat'][i].float(),
                       'ctx_feat': cache['ctx_feat'][i].float()}
                loss = crit(o, tgt, y)['loss']
            else:
                loss = crit(o['logits'], y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item(); nb += 1
        sch.step()
        vm = metrics(y_val, predict(model, val_ds, C, device).argmax(1))
        improved = vm['mf1'] > st['best']
        if improved:
            st.update(best=vm['mf1'], best_epoch=ep, patience=0)
            torch.save(model.state_dict(), best_p)
        else:
            st['patience'] += 1
        st['epoch'] = ep
        st['history'].append(dict(epoch=ep, loss=tot / max(nb, 1), val_acc=vm['acc'], val_mf1=vm['mf1'],
                                  val_n1=vm['f1'][1], sec=time.time() - t0))
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sch=sch.state_dict(), state=st), last_p)
        print(f'    ep{ep:3d} loss {tot / max(nb, 1):.4f} | val acc {vm["acc"]:.4f} mf1 {vm["mf1"]:.4f} '
              f'n1 {vm["f1"][1]:.4f} | {time.time() - t0:.0f}s {"*" if improved else ""}')
        if st['patience'] >= C['PATIENCE']:
            print(f'    early stop (best epoch {st["best_epoch"]})')
            break

    # ---- test with the best-validation checkpoint
    model.load_state_dict(torch.load(best_p, map_location=device, weights_only=True))
    logits = predict(model, test_ds, C, device)
    tm = metrics(y_test, logits.argmax(1))
    pred = dict(rec=test_ds.rec_ids(), center=test_ds.index[:, 1], y=y_test, logits=logits.astype(np.float16))
    tfm = None
    if store.has_full():
        full_ds = SeqDS(store, te, which='full')
        lf = predict(model, full_ds, C, device)
        tfm = metrics(full_ds.labels(), lf.argmax(1))
        pred.update(full_rec=full_ds.rec_ids(), full_center=full_ds.index[:, 1], full_y=full_ds.labels(),
                    full_logits=lf.astype(np.float16))
    np.savez_compressed(os.path.join(odir, 'preds_test.npz'), **pred)
    res = dict(arm=name, kind=arm['kind'], loss=arm['loss'], src=src_name,
               src_seed=src_seed if src_name else None, T=arm.get('T'), lr=lr, ls=ls, bs=arm['bs'],
               selected_from=arm.get('selected_from'), tuning=arm.get('tuning'),
               fold=fold, seed=seed, target_dim=target_dim, params_total=p_total, params_inference=p_infer,
               best_epoch=st['best_epoch'], best_val_mf1=st['best'], epochs_run=st['epoch'],
               n_train=len(train_ds), n_val=len(val_ds), n_test=len(test_ds),
               test=tm, test_full_night=tfm, minutes=(time.time() - t_start) / 60, history=st['history'])
    json.dump(res, open(os.path.join(odir, 'result.json'), 'w'), indent=1)
    if os.path.exists(last_p):
        os.remove(last_p)
    print(f'  ==> {name} fold{fold}: acc {tm["acc"]:.4f} mf1 {tm["mf1"]:.4f} kappa {tm["kappa"]:.4f} '
          f'n1 {tm["f1"][1]:.4f} ({res["minutes"]:.0f} min)')
    return res


# ================================================================ aggregation & statistics
def collect(C):
    rows = []
    for a in C['ARMS']:
        for s in C['SEEDS']:
            for f in range(C['K_FOLDS']):
                r = load_result(C, a['name'], f, s)
                if r is None:
                    continue
                for split in ['test', 'test_full_night']:
                    m = r.get(split)
                    if m is None:
                        continue
                    row = dict(arm=a['name'], seed=s, fold=f, split=split, src=r['src'],
                               acc=m['acc'], mf1=m['mf1'], kappa=m['kappa'],
                               params_total=r['params_total'], params_inference=r['params_inference'],
                               best_epoch=r['best_epoch'], minutes=r['minutes'])
                    row.update({f'f1_{nm}': v for nm, v in zip(STAGES, m['f1'])})
                    rows.append(row)
    return pd.DataFrame(rows)


def summary(df, split='test'):
    d = df[df.split == split]
    cols = ['acc', 'mf1', 'kappa'] + [f'f1_{s}' for s in STAGES]
    g = d.groupby('arm')[cols]
    mean, sd = g.mean(), g.std(ddof=1)
    out = pd.DataFrame(index=mean.index)
    for c in cols:
        scale = 1 if c == 'kappa' else 100
        out[c] = [f'{m * scale:.2f} ± {s * scale:.2f}' if scale == 100 else f'{m:.4f} ± {s:.4f}'
                  for m, s in zip(mean[c], sd[c].fillna(0))]
    out['n_runs'] = g.size()
    return out


def _subject_scores(y, p, subj):
    rows = []
    for s in np.unique(subj):
        m = subj == s
        present = np.unique(y[m])
        f1 = f1_score(y[m], p[m], labels=list(range(5)), average=None, zero_division=0)
        rows.append(dict(subject=s, acc=float((y[m] == p[m]).mean()),
                         mf1=float(f1_score(y[m], p[m], labels=present, average='macro', zero_division=0)),
                         kappa=float(cohen_kappa_score(y[m], p[m])),
                         n1=float(f1[1]) if (y[m] == 1).any() else np.nan, n=int(m.sum())))
    return pd.DataFrame(rows)


def subject_table(C, arm, seed, full_night=False):
    """Per-subject metrics pooled over both nights; Macro-F1 over stages present in that subject."""
    ys, ps, ss = [], [], []
    for f in range(C['K_FOLDS']):
        name = resolve_source(C, arm, f, seed)
        p = os.path.join(run_dir(C, name, f, seed), 'preds_test.npz')
        if not os.path.exists(p):
            return None
        d = np.load(p, allow_pickle=True)
        k = 'full_' if full_night else ''
        ys.append(d[f'{k}y']); ps.append(d[f'{k}logits'].astype(np.float32).argmax(1))
        ss.append(np.array([r[:5] for r in d[f'{k}rec']]))
    return _subject_scores(np.concatenate(ys), np.concatenate(ps), np.concatenate(ss))


def _bh(p):
    p = np.asarray(p, float)
    n = len(p)
    order = np.argsort(p)
    q = np.empty(n)
    q[order] = np.minimum.accumulate((p[order] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1)


def subject_wilcoxon(C, pairs, metrics_=('mf1', 'acc', 'n1'), n_boot=10000, full_night=False):
    """SENSITIVITY analysis: paired per-subject metrics (Wilcoxon + bootstrap of the mean difference)."""
    from scipy.stats import wilcoxon
    rng = np.random.default_rng(0)
    rows = []
    for a, b in pairs:
        ta = [subject_table(C, a, s, full_night) for s in C['SEEDS']]
        tb = [subject_table(C, b, s, full_night) for s in C['SEEDS']]
        if any(t is None for t in ta + tb):
            print(f'  (skip {a} vs {b}: runs incomplete)')
            continue
        A = pd.concat(ta).groupby('subject').mean(numeric_only=True)     # mean over seeds per subject
        B = pd.concat(tb).groupby('subject').mean(numeric_only=True)
        for m in metrics_:
            d = (A[m] - B[m]).dropna()
            boot = rng.choice(d.values, size=(n_boot, len(d)), replace=True).mean(1)
            w = wilcoxon(d.values) if (d != 0).any() else None
            rows.append(dict(contrast=f'{a} - {b}', metric=m, n_subjects=len(d),
                             mean_diff_pp=100 * d.mean(), ci95_low_pp=100 * np.percentile(boot, 2.5),
                             ci95_high_pp=100 * np.percentile(boot, 97.5), subjects_better=int((d > 0).sum()),
                             p_wilcoxon=w.pvalue if w is not None else np.nan))
    out = pd.DataFrame(rows)
    if len(out):
        out['q_bh'] = _bh(out['p_wilcoxon'].fillna(1).values)
    return out


# ================================================================ PRIMARY analysis (pre-specified)
# Metric: Macro-F1 of all pooled out-of-fold test predictions (5 folds; pooled over training seeds).
# Inference: subject-cluster bootstrap (whole subjects resampled with replacement, both nights kept together),
# 95% percentile CI of the difference, bootstrap p; BH-FDR within the primary family (one metric x all KD contrasts).
# Secondary metrics (accuracy, kappa, N1-F1) use the same bootstrap and form a separate BH family.
def _cm_metrics(cm):
    """cm: (..., 5, 5) rows = true, cols = predicted."""
    cm = cm.astype(np.float64)
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    rows, cols = cm.sum(-1), cm.sum(-2)
    n = cm.sum((-1, -2))
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rows + cols > 0, 2 * tp / (rows + cols), 0.0)
        acc = tp.sum(-1) / n
        pe = (rows * cols).sum(-1) / n ** 2
        kappa = (acc - pe) / (1 - pe)
    return dict(mf1=f1.mean(-1), acc=acc, kappa=kappa, n1=f1[..., 1])


def subject_cms(C, arm, full_night=False, exclude_subjects=None):
    """{subject: 5x5 confusion matrix} over all folds and seeds, plus the label vector (to check alignment).
    exclude_subjects: subjects dropped from every arm (sensitivity analyses)."""
    cms, y_ref = {}, None
    k = 'full_' if full_night else ''
    excl = set(exclude_subjects or ())
    for s in C['SEEDS']:
        ys = []
        for f in range(C['K_FOLDS']):
            p = os.path.join(run_dir(C, arm, f, s), 'preds_test.npz')
            if not os.path.exists(p):
                return None, None
            d = np.load(p, allow_pickle=True)
            subj = np.array([r[:5] for r in d[f'{k}rec']])
            keep = ~np.isin(subj, list(excl))
            y = d[f'{k}y'].astype(np.int64)[keep]
            pr = d[f'{k}logits'].astype(np.float32).argmax(1)[keep]
            subj = subj[keep]
            for sj in np.unique(subj):
                m = subj == sj
                cm = np.zeros((5, 5), np.int64)
                np.add.at(cm, (y[m], pr[m]), 1)
                cms[sj] = cms.get(sj, 0) + cm
            ys.append(y)
        y_ref = np.concatenate(ys) if y_ref is None else y_ref
    return cms, y_ref


def cluster_contrasts(C, pairs, n_boot=10000, primary='mf1', full_night=False, seed=0, exclude_subjects=None):
    rng = np.random.default_rng(seed)
    rows = []
    for a, b in pairs:
        A, ya = subject_cms(C, a, full_night, exclude_subjects)
        B, yb = subject_cms(C, b, full_night, exclude_subjects)
        if A is None or B is None:
            print(f'  (skip {a} vs {b}: runs incomplete)')
            continue
        assert np.array_equal(ya, yb), f'{a} and {b} were not evaluated on identical test epochs'
        subs = sorted(A)
        CA, CB = np.stack([A[s] for s in subs]), np.stack([B[s] for s in subs])
        W = rng.multinomial(len(subs), np.full(len(subs), 1 / len(subs)), size=n_boot)   # (n_boot, S) draws
        ma, mb = _cm_metrics(CA.sum(0)), _cm_metrics(CB.sum(0))
        ba = _cm_metrics(np.einsum('bs,sij->bij', W, CA))
        bb = _cm_metrics(np.einsum('bs,sij->bij', W, CB))
        for met in ('mf1', 'acc', 'kappa', 'n1'):
            d = ba[met] - bb[met]
            p = min(1.0, 2 * (min((d <= 0).sum(), (d >= 0).sum()) + 1) / (n_boot + 1))
            rows.append(dict(contrast=f'{a} - {b}', metric=met, family='primary' if met == primary else 'secondary',
                             value_a=float(ma[met]), value_b=float(mb[met]), diff_pp=100 * float(ma[met] - mb[met]),
                             ci95_low_pp=100 * float(np.percentile(d, 2.5)),
                             ci95_high_pp=100 * float(np.percentile(d, 97.5)),
                             p_boot=p, n_subjects=len(subs), n_epochs=int(CA.sum())))
    out = pd.DataFrame(rows)
    if len(out):
        out['q_bh'] = np.nan
        for fam in out.family.unique():
            m = out.family == fam
            out.loc[m, 'q_bh'] = _bh(out.loc[m, 'p_boot'].values)
    return out


def tuned_dependent_arms(arms):
    """Arms whose test results depend on the tuned configuration: arms with 'tune' and every arm distilled from one."""
    dep = set()
    for a in arms:                                 # list order = dependency order
        if 'tune' in a or a.get('src') in dep:
            dep.add(a['name'])
    return dep


def hybrid_contrasts(ct_all, ct_excl, dependent):
    """Primary inference (chosen 2026-10-15): a contrast involving a tuned-dependent arm takes its row from the
    analysis without the tuning subjects (ct_excl), every other contrast from the full analysis (ct_all).
    Rows are copied unchanged; BH-FDR is recomputed within each family over the combined set."""
    def uses_dep(c):
        return any(x.strip() in dependent for x in c.split(' - '))
    out = pd.concat([ct_excl[ct_excl.contrast.map(uses_dep)].assign(subject_set='excl. tuning subjects'),
                     ct_all[~ct_all.contrast.map(uses_dep)].assign(subject_set='all subjects')])
    order = {c: i for i, c in enumerate(ct_all.contrast.unique())}
    out = out.sort_values(['family', 'contrast'], key=lambda s: s.map(order) if s.name == 'contrast' else s)
    out['q_bh'] = np.nan
    for fam in out.family.unique():
        m = out.family == fam
        out.loc[m, 'q_bh'] = _bh(out.loc[m, 'p_boot'].values)
    return out.reset_index(drop=True)


def pooled_table(C, arms, full_night=False, exclude_subjects=None):
    """Pooled out-of-fold metrics per arm (the numbers the primary analysis is about)."""
    rows = []
    for a in arms:
        cms, _ = subject_cms(C, a, full_night, exclude_subjects)
        if cms is None:
            continue
        m = _cm_metrics(sum(cms.values()))
        f1 = _cm_f1_per_class(sum(cms.values()))
        rows.append(dict(arm=a, mf1=100 * m['mf1'], acc=100 * m['acc'], kappa=m['kappa'],
                         **{f'f1_{s}': 100 * v for s, v in zip(STAGES, f1)}, n_subjects=len(cms)))
    return pd.DataFrame(rows).set_index('arm')


def _cm_f1_per_class(cm):
    cm = cm.astype(np.float64)
    tp = np.diag(cm)
    den = cm.sum(1) + cm.sum(0)
    return np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)


def tuning_table(C):
    rows = []
    seed0 = C['SEEDS'][0]
    for a in C['ARMS']:
        if 'folds' not in a:
            continue
        for f in a['folds']:
            r = load_result(C, a['name'], f, seed0)
            if r is not None:
                rows.append(dict(arm=a['name'], fold=f, lr=r['lr'], ls=r.get('ls'), best_val_mf1=r['best_val_mf1'],
                                 best_epoch=r['best_epoch'], epochs_run=r['epochs_run']))
    return pd.DataFrame(rows)


# ================================================================ cross-dataset (zero-shot)
def eval_cross(C, arm, seed, tgt_store, tgt_recs, tgt_name, device):
    """Each fold model of `arm` (trained in C) predicts every target recording (trimmed targets). Resumable."""
    rows = []
    ds = SeqDS(tgt_store, tgt_recs)
    y = ds.labels()
    for f in range(C['K_FOLDS']):
        odir = run_dir(C, arm, f, seed)
        jp = os.path.join(odir, f'cross_{tgt_name}.json')
        if not os.path.exists(jp):
            r = load_result(C, arm, f, seed)
            if r is None:
                print(f'  (skip {arm} fold{f}: not trained)')
                continue
            model = build_model(r['kind'], r['target_dim']).to(device)
            model.load_state_dict(torch.load(os.path.join(odir, 'best.pt'), map_location=device, weights_only=True))
            logits = predict(model, ds, C, device)
            m = metrics(y, logits.argmax(1))
            np.savez_compressed(os.path.join(odir, f'cross_{tgt_name}.npz'), rec=ds.rec_ids(), center=ds.index[:, 1],
                                y=y, logits=logits.astype(np.float16))
            json.dump(m, open(jp, 'w'), indent=1)
        m = json.load(open(jp))
        rows.append(dict(arm=arm, seed=seed, fold=f, target=tgt_name, acc=m['acc'], mf1=m['mf1'], kappa=m['kappa'],
                         **{f'f1_{s}': v for s, v in zip(STAGES, m['f1'])}))
    return pd.DataFrame(rows)

In [ ]:
%%writefile /content/skd_code/xai.py
"""XAI v2 on the DREAMS K-complex and sleep-spindle databases (R1 attribution comments, R2.2).

Data facts (DREAMS documentation, Zenodo 10.5281/zenodo.2650142):
  * K-complexes: 10 excerpts of 30 min, healthy subjects, CZ-A1 @ 200 Hz.
  * Spindles: 8 excerpts of 30 min, patients, C3-A1 (excerpts 1, 3) or CZ-A1, 100 / 200 / 50 Hz.
  * Hypnogram_excerptN.txt: one value per 5 s, R&K: 5 W, 4 REM, 3 S1, 2 S2, 1 S3, 0 S4, -1 movement, -2/-3 unknown.
  * Visual_scoring1_excerptN.txt: expert-1 events (onset s, duration s); expert 1 scored every excerpt.
  * excerptN.txt: first line = name of the scored channel.
Preprocessing is identical to training (prep.B_BP filter per epoch, per-epoch per-channel z-score, fp16 storage);
the single scored channel is replicated into the 3 input slots (EEG, EEG, EOG), i.e. out of the training distribution.
"""
import os, re, glob, hashlib, subprocess, urllib.request
import numpy as np, pandas as pd, scipy.signal, torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import prep, skd

DBS = ('kcomplexes', 'spindles')
EVENT_NAME = {'kcomplexes': 'K-complex', 'spindles': 'Spindle'}
ZENODO = {'kcomplexes': ('https://zenodo.org/api/records/2650142/files/DatabaseKcomplexes.rar/content',
                         '0bfc5bde8e36bcb1edcded50077423de'),
          'spindles': ('https://zenodo.org/api/records/2650142/files/DatabaseSpindles.rar/content',
                       '2f8a101194e133dd4324f21047dce579')}
RK_TO_AASM = {5: 0, 3: 1, 2: 2, 1: 3, 0: 3, 4: 4}      # S3 + S4 -> N3; -1 / -2 / -3 -> not a stage
HYP_S = 5                                             # seconds per hypnogram value
SIGMA_BAND = (11.0, 16.0)
B_SIG, A_SIG = scipy.signal.butter(N=4, Wn=SIGMA_BAND, btype='bandpass', fs=prep.FS)
CENTER = skd.SEQ_LEN // 2


# ================================================================ data
def _md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()


def download_dreams(raw_dir):
    """Download the two DREAMS archives from Zenodo (md5-checked) and extract them. Returns {db: folder}."""
    os.makedirs(raw_dir, exist_ok=True)
    out = {}
    for db, (url, md5) in ZENODO.items():
        rar = os.path.join(raw_dir, f'{db}.rar')
        if not (os.path.exists(rar) and _md5(rar) == md5):
            urllib.request.urlretrieve(url, rar)
        assert _md5(rar) == md5, f'{db}: md5 mismatch'
        dst = os.path.join(raw_dir, db)
        os.makedirs(dst, exist_ok=True)
        subprocess.run(['unrar', 'x', '-o+', '-inul', rar, dst + os.sep], check=True)
        out[db] = find_excerpt_dir(dst)
    return out


def find_excerpt_dir(root):
    hits = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, '**', 'excerpt*.edf'), recursive=True)})
    assert len(hits) == 1, f'expected one folder with excerpt*.edf under {root}, got {hits}'
    return hits[0]


def _two_col(path):
    rows = []
    for line in open(path, errors='ignore'):
        parts = line.split()
        if len(parts) >= 2 and not line.lstrip().startswith('['):
            rows.append((float(parts[0]), float(parts[1])))
    return rows


def _hypnogram(path):
    return np.array([int(float(l)) for l in open(path, errors='ignore') if l.strip() and not l.strip().startswith('[')])


def _epoch_stage(hyp_vals):
    """Majority AASM stage of the six 5-s values of one 30-s epoch; -1 if no valid value or a tie."""
    st = [RK_TO_AASM[v] for v in hyp_vals if v in RK_TO_AASM]
    if not st:
        return -1
    c = np.bincount(st, minlength=5)
    return int(c.argmax()) if (c == c.max()).sum() == 1 else -1


def process_excerpt(db, folder, n, out_dir):
    """One excerpt -> <out_dir>/<db>_excerptN.npz in the skd.Store format (+ event mask). Returns a metadata row."""
    import mne
    first = open(os.path.join(folder, f'excerpt{n}.txt'), errors='ignore').readline().strip()
    ch = first.strip('[]').strip()
    raw = mne.io.read_raw_edf(os.path.join(folder, f'excerpt{n}.edf'), preload=False, verbose='ERROR')
    extras = raw._raw_extras[0]
    native = dict(zip(raw.ch_names, extras['n_samps'] / extras['record_length'][0]))
    fs_native = float(native[ch])
    assert fs_native == raw.info['sfreq'], f'{db} {n}: scored channel is not at the file rate'
    x = raw.get_data(picks=[ch])[0] * 1e6                                   # µV at the native rate
    up, down = int(prep.FS), int(fs_native)
    g = np.gcd(up, down)
    if up != down:
        x = scipy.signal.resample_poly(x, up // g, down // g)
    n_ep = len(x) // prep.EP
    x = x[:n_ep * prep.EP].reshape(n_ep, 1, prep.EP)
    x = np.repeat(x, 3, axis=1)                                             # one channel -> 3 input slots
    x = scipy.signal.filtfilt(prep.B_BP, prep.A_BP, x, axis=2)              # identical to training (_finalize)
    x = (x - x.mean(axis=2, keepdims=True)) / (x.std(axis=2, keepdims=True) + 1e-6)

    hyp = _hypnogram(os.path.join(folder, f'Hypnogram_excerpt{n}.txt'))
    per = prep.EPOCH_S // HYP_S
    y = np.array([_epoch_stage(hyp[i * per:(i + 1) * per]) for i in range(n_ep)], dtype=np.int64)

    events = _two_col(os.path.join(folder, f'Visual_scoring1_excerpt{n}.txt'))
    mask = np.zeros(n_ep * prep.EP, bool)
    for on, du in events:
        a, b = int(round(on * prep.FS)), int(round((on + du) * prep.FS))
        mask[max(a, 0):min(b, len(mask))] = True
    mask = mask.reshape(n_ep, prep.EP)
    ev_ep = np.where(mask.any(1) & ~mask.all(1))[0]                          # targets: epochs with an event

    rid = f'{db}_excerpt{n}'
    np.savez(os.path.join(out_dir, f'{rid}.npz'), x=x.astype(np.float16), y=y, mask=mask,
             centers_trim=ev_ep.astype(np.int64), centers_full=np.arange(n_ep, dtype=np.int64))
    stages = {s: int((y == i).sum()) for i, s in enumerate(skd.STAGES)}
    return dict(rec=rid, db=db, excerpt=int(n), channel=ch, fs_native=fs_native, resample=f'{up // g}/{down // g}',
                n_samples_native=int(raw.n_times), n_epochs=n_ep, n_events=len(events), n_event_epochs=len(ev_ep),
                event_samples_frac=float(mask[ev_ep].mean()) if len(ev_ep) else np.nan,
                hyp_values=len(hyp), **{f'stage_{s}': v for s, v in stages.items()}, stage_unscored=int((y < 0).sum()))


def process_all(dirs, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for db in DBS:
        nums = sorted(int(m.group(1)) for p in glob.glob(os.path.join(dirs[db], 'excerpt*.edf'))
                      if (m := re.search(r'excerpt(\d+)\.edf$', p)))
        rows += [process_excerpt(db, dirs[db], k, out_dir) for k in nums]
    return pd.DataFrame(rows)


# ================================================================ attribution
def _target_logit(model, x, cls):
    return model(x)['logits'].gather(1, cls[:, None]).squeeze(1)


def ig_grid(kind, steps, lo=1e-6):
    """Path points alpha in [0, 1] and trapezoid weights. 'uniform': midpoint rule (the usual IG implementation).
    'log': 0 plus log-spaced points 1e-6..1 — the models normalise early (GroupNorm / LayerNorm), so the target logit
    changes almost entirely at very small alpha, which a uniform grid does not resolve (checked: completeness)."""
    if kind == 'uniform':
        return (np.arange(steps) + 0.5) / steps, np.full(steps, 1.0 / steps)
    a = np.concatenate([[0.0], np.logspace(np.log10(lo), 0, steps - 1)])
    w = np.zeros_like(a)
    d = np.diff(a)
    w[:-1] += d / 2
    w[1:] += d / 2
    return a, w


def attribute(model, x, ig_steps=64, ig_kind='log', ig_lo=1e-6):
    """x: (B, 21, 3, 3000) float32 on device. Target = logit of the class predicted for the clean input.
    Returns dict of (B, 3000) maps for the CENTRE epoch (|.| summed over the 3 input slots) + IG completeness
    |sum(IG) - (F(x) - F(0))| / |F(x) - F(0)| (sum over the whole 21-epoch input)."""
    use_cudnn = not hasattr(model, 'gru')            # cuDNN RNN backward needs train mode; we keep eval mode
    with torch.backends.cudnn.flags(enabled=use_cudnn):
        with torch.no_grad():
            logits = model(x)['logits']
            cls = logits.argmax(1)
            f_x = logits.gather(1, cls[:, None]).squeeze(1)
            f_0 = _target_logit(model, torch.zeros_like(x), cls)
        xg = x.clone().requires_grad_(True)
        g, = torch.autograd.grad(_target_logit(model, xg, cls).sum(), xg)
        grad_map = g[:, CENTER].abs().sum(1)
        acc = torch.zeros_like(x)
        for a, w in zip(*ig_grid(ig_kind, ig_steps, ig_lo)):  # zero baseline
            xa = (x * float(a)).requires_grad_(True)
            gk, = torch.autograd.grad(_target_logit(model, xa, cls).sum(), xa)
            acc += float(w) * gk
        ig = x * acc
        ig_map = ig[:, CENTER].abs().sum(1)
        completeness = (ig.sum((1, 2, 3)) - (f_x - f_0)).abs() / ((f_x - f_0).abs() + 1e-8)
    return dict(gradient=grad_map.detach().cpu().numpy(), ig=ig_map.detach().cpu().numpy(),
                pred=cls.cpu().numpy(), ig_rel_err=completeness.detach().cpu().numpy())


def baselines(xc):
    """Model-free maps on the centre epoch actually given to the models (first input slot). xc: (B, 3000)."""
    env = np.abs(scipy.signal.hilbert(xc, axis=1))
    sig = np.abs(scipy.signal.hilbert(scipy.signal.filtfilt(B_SIG, A_SIG, xc, axis=1), axis=1))
    return {'abs_x': np.abs(xc), 'envelope': env, 'sigma_power': sig ** 2}


def _auroc(maps, mask):
    return np.array([roc_auc_score(m, s) for s, m in zip(maps, mask)])


def spec_ready(runs_dir, arm, k_folds, seed):
    """True when every fold model of this configuration has been trained (result.json + best.pt present)."""
    C = dict(RUNS_DIR=runs_dir)
    return all(os.path.exists(os.path.join(skd.run_dir(C, arm, f, seed), n))
               for f in range(k_folds) for n in ('result.json', 'best.pt'))


def load_model(runs_dir, arm, fold, seed, device):
    C = dict(RUNS_DIR=runs_dir)
    res = skd.load_result(C, arm, fold, seed)
    assert res is not None, f'{runs_dir}: {arm} fold{fold} seed{seed} not trained'
    m = skd.build_model(res['kind'], res['target_dim']).to(device)
    m.load_state_dict(torch.load(os.path.join(skd.run_dir(C, arm, fold, seed), 'best.pt'), map_location=device,
                                 weights_only=True))
    return m.eval(), res['kind']


def random_model(kind, seed, device):
    torch.manual_seed(seed)
    return skd.build_model(kind).to(device).eval()


def run_model(model, store, recs, mask_of, device, batch=32, ig_steps=64, ig_lo=1e-6):
    """Per-epoch AUROC rows for every target epoch of every excerpt."""
    rows = []
    for rec in recs:
        ds = skd.SeqDS(store, [rec])
        centers = ds.index[:, 1]
        out = {'gradient': [], 'ig': [], 'pred': [], 'ig_rel_err': []}
        for xb, _, _ in DataLoader(ds, batch_size=batch, shuffle=False):
            a = attribute(model, xb.to(device), ig_steps, 'log', ig_lo)
            for k in out:
                out[k].append(a[k])
        out = {k: np.concatenate(v) for k, v in out.items()}
        m = mask_of[rec][centers]
        y = store.recs[store.pos[rec]]['y'][centers]
        for meth in ('gradient', 'ig'):
            au = _auroc(out[meth], m)
            rows += [dict(rec=rec, epoch=int(c), method=meth, auroc=float(v), pred=int(p), stage=int(s),
                          ig_rel_err=float(e)) for c, v, p, s, e in zip(centers, au, out['pred'], y, out['ig_rel_err'])]
    return pd.DataFrame(rows)


def run_baselines(store, recs, mask_of):
    rows = []
    for rec in recs:
        r = store.recs[store.pos[rec]]
        centers = r['trim']
        xc = r['x'][centers, 0].astype(np.float32)
        m = mask_of[rec][centers]
        for name, maps in baselines(xc).items():
            rows += [dict(rec=rec, epoch=int(c), method=name, auroc=float(v)) for c, v in zip(centers, _auroc(maps, m))]
    return pd.DataFrame(rows)


# ================================================================ statistics (unit = excerpt; plan section 0.4)
def excerpt_level(df, by):
    """Mean AUROC over target epochs within an excerpt, then mean over the fold / random models."""
    e = df.groupby(by + ['model_id', 'rec'], as_index=False).auroc.mean()
    return e.groupby(by + ['rec'], as_index=False).auroc.mean()


def _boot_median_ci(v, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    b = np.median(rng.choice(v, size=(n_boot, len(v)), replace=True), axis=1)
    return np.percentile(b, 2.5), np.percentile(b, 97.5)


def wilcoxon_row(d, **info):
    from scipy.stats import wilcoxon
    d = np.asarray(d, float)
    lo, hi = _boot_median_ci(d)
    p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return dict(**info, n_excerpts=len(d), median=float(np.median(d)), ci95_low=float(lo), ci95_high=float(hi),
                n_positive=int((d > 0).sum()), p_wilcoxon=float(p))


def bh_by_family(tab):
    tab = tab.copy()
    tab['q_bh'] = np.nan
    for fam, idx in tab.groupby('family').groups.items():
        tab.loc[idx, 'q_bh'] = skd._bh(tab.loc[idx, 'p_wilcoxon'].values)
    return tab


BASELINES = {'kcomplexes': ['abs_x', 'envelope'], 'spindles': ['abs_x', 'envelope', 'sigma_power']}
METHODS = ('gradient', 'ig')


def planned_tests(specs, random_of):
    """The complete list of tests, fixed before any model is evaluated (plan section 0.4).
    Q1: trained-model AUROC vs 0.5. Q2: trained model vs each model-free baseline. Q3: trained vs random-init.
    One BH family per (question, event database)."""
    rows = []
    for db in DBS:
        for s in specs:
            for m in METHODS:
                rows.append(dict(family=f'Q1|{db}', db=db, spec=s, method=m, versus='chance (0.5)'))
                rows += [dict(family=f'Q2|{db}', db=db, spec=s, method=m, versus=b) for b in BASELINES[db]]
                rows.append(dict(family=f'Q3|{db}', db=db, spec=s, method=m, versus=random_of[s]))
    return pd.DataFrame(rows)


def run_tests(plan, exc_models, exc_base):
    """exc_models: excerpt-level AUROC with columns spec, method, db, rec, auroc (trained + random specs);
    exc_base: same for baselines (spec = baseline name, method = baseline name)."""
    M = exc_models.set_index(['spec', 'method', 'db', 'rec']).auroc
    B = exc_base.set_index(['method', 'db', 'rec']).auroc
    out = []
    for t in plan.itertuples():
        a = M.loc[(t.spec, t.method, t.db)]
        if t.versus == 'chance (0.5)':
            d = a - 0.5
        elif t.versus in B.index.get_level_values(0):
            d = a - B.loc[(t.versus, t.db)].reindex(a.index)
        else:
            d = a - M.loc[(t.versus, t.method, t.db)].reindex(a.index)
        assert d.notna().all() and len(d) == (10 if t.db == 'kcomplexes' else 8), (t, len(d))
        out.append(wilcoxon_row(d.values, family=t.family, db=t.db, spec=t.spec, method=t.method, versus=t.versus,
                                median_auroc=float(a.median())))
    return bh_by_family(pd.DataFrame(out))


def completeness_check(model, store, rec, device, n=32):
    """Median IG completeness error on the first n target epochs of one excerpt, uniform-50 vs log-64 grids."""
    ds = skd.SeqDS(store, [rec])
    x = torch.stack([ds[i][0] for i in range(min(n, len(ds)))]).to(device)
    return {f'{k}{s}': float(np.median(attribute(model, x, s, k)['ig_rel_err'])) for k, s in [('uniform', 50), ('log', 64)]}


def figure_epoch(store, mask_of, db, meta):
    """Pre-specified example: the excerpt with the most expert events; in it, the target epoch with the largest
    event fraction (ties -> earliest)."""
    rec = meta[meta.db == db].sort_values(['n_events', 'excerpt'], ascending=[False, True]).rec.iloc[0]
    r = store.recs[store.pos[rec]]
    frac = mask_of[rec][r['trim']].mean(1)
    return rec, int(r['trim'][int(np.argmax(frac))])


def plot_auroc(exc_m, exc_b, specs, random_specs, out_png):
    """Fig. 2b: excerpt-level AUROC (one dot per excerpt, bar = median) for every trained model, the randomly
    initialised models and the model-free baselines; one panel per event type; gradient and IG side by side."""
    import matplotlib.pyplot as plt
    rows = [(s, m) for s in list(specs) + list(random_specs) for m in METHODS]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
    for ax, db in zip(axes, DBS):
        items = [(f'{s}\n{m}', exc_m[(exc_m.spec == s) & (exc_m.method == m) & (exc_m.db == db)].auroc.values)
                 for s, m in rows]
        items += [(b, exc_b[(exc_b.method == b) & (exc_b.db == db)].auroc.values) for b in BASELINES[db]]
        for i, (lab, v) in enumerate(items):
            col = 'tab:gray' if 'random' in lab else ('tab:green' if '\n' not in lab else
                                                      ('tab:blue' if lab.endswith('ig') else 'tab:orange'))
            ax.bar(i, np.median(v), color=col, alpha=0.35, width=0.7)
            ax.scatter(np.full(len(v), i) + np.linspace(-0.2, 0.2, len(v)), v, s=8, color=col, zorder=3)
        ax.axhline(0.5, color='k', lw=0.8, ls='--')
        ax.set_xticks(range(len(items)))
        ax.set_xticklabels([l for l, _ in items], rotation=90, fontsize=6)
        ax.set_title(f'{EVENT_NAME[db]} (n = {len(items[0][1])} excerpts)', fontsize=9)
    axes[0].set_ylabel('AUROC (excerpt mean over fold models)', fontsize=8)
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_png.replace('.png', '.pdf'))
    return fig


def plot_examples(panels, out_png):
    """panels: list of dict(title, x (3000,), mask (3000,), maps {label: (3000,)}). Time axis in seconds."""
    import matplotlib.pyplot as plt
    t = np.arange(prep.EP) / prep.FS
    nrow = max(len(p['maps']) for p in panels) + 1
    fig, axes = plt.subplots(nrow, len(panels), figsize=(6.5 * len(panels), 1.25 * nrow), sharex=True, squeeze=False)
    for j, p in enumerate(panels):
        spans = np.flatnonzero(np.diff(np.r_[0, p['mask'].astype(int), 0]))
        for i, (lab, y) in enumerate([('Input (z-scored)', p['x'])] + list(p['maps'].items())):
            ax = axes[i, j]
            for a, b in zip(spans[::2], spans[1::2]):
                ax.axvspan(a / prep.FS, b / prep.FS, color='tab:orange', alpha=0.25, lw=0)
            ax.plot(t, y, lw=0.6, color='k' if i == 0 else 'tab:blue')
            ax.set_ylabel(lab, fontsize=7, rotation=0, ha='right', va='center')
            ax.tick_params(labelsize=7)
            ax.set_yticks([])
        axes[0, j].set_title(p['title'], fontsize=9)
        axes[-1, j].set_xlabel('Time within the 30-s epoch (s)', fontsize=8)
        axes[-1, j].set_xticks(np.arange(0, 31, 5))
    fig.tight_layout()
    fig.savefig(out_png, dpi=200)
    fig.savefig(out_png.replace('.png', '.pdf'))
    return fig

In [ ]:
%%writefile /content/skd_code/cka.py
"""CKA v2 (R1: CKA baselines; plan C1, C3). Linear CKA between the centre-epoch representations of two models,
computed per test subject (unit = subject; both models are the fold model that holds that subject out).
Levels: 'spatial' = CNN embedding of the centre epoch (cnn_feat), 'temporal' = context representation used by the
classifier (ctx_feat: Transformer / linear-attention output after LayerNorm, GRU output after LayerNorm).
Baseline: the same comparison with the second model replaced by a randomly initialised model of its architecture."""
import itertools, os
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
import skd

LEVELS = {'spatial': 'cnn_feat', 'temporal': 'ctx_feat'}


def linear_cka(X, Y):
    X = X.astype(np.float64) - X.mean(0, keepdims=True)
    Y = Y.astype(np.float64) - Y.mean(0, keepdims=True)
    num = np.linalg.norm(X.T @ Y) ** 2
    den = np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y)
    return float(num / den) if den > 0 else np.nan


@torch.no_grad()
def features(model, ds, device, bs=256):
    model.eval()
    out = {lv: [] for lv in LEVELS}
    for x, _, _ in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=0 if os.name == 'nt' else 2):
        o = model(x.to(device))
        for lv, key in LEVELS.items():
            out[lv].append(o[key].float().cpu().numpy())
    return {lv: np.concatenate(v) for lv, v in out.items()}


def trained_model(runs_dir, arm, fold, seed, device):
    C = dict(RUNS_DIR=runs_dir)
    res = skd.load_result(C, arm, fold, seed)
    assert res is not None, f'{arm} fold{fold} not trained'
    m = skd.build_model(res['kind'], res['target_dim']).to(device)
    m.load_state_dict(torch.load(f"{skd.run_dir(C, arm, fold, seed)}/best.pt", map_location=device, weights_only=True))
    return m, res['kind']


def random_model(kind, seed, device):
    torch.manual_seed(seed)
    return skd.build_model(kind).to(device)


def planned_tests(models, kind_of):
    """Fixed before computing any CKA. 'vs_random': each trained pair (X, Y) vs (X, random-init of Y's architecture).
    'pathway': the comparisons the manuscript interprets. One BH family per (test type, level)."""
    rows = []
    for lv in LEVELS:
        for x, y in itertools.combinations(models, 2):
            rows.append(dict(family=f'vs_random|{lv}', level=lv, a=f'{x} ~ {y}', b=f'{x} ~ random {kind_of[y]}'))
        for a, b in [('Assistant ~ Edge-Staged', 'Assistant ~ Edge-Direct'),
                     ('Teacher ~ Edge-Direct', 'Teacher ~ Edge-Staged'),
                     ('Assistant ~ Edge-Staged', 'Assistant ~ Edge-Scratch'),
                     ('Assistant ~ Edge-Direct', 'Assistant ~ Edge-Scratch')]:
            rows.append(dict(family=f'pathway|{lv}', level=lv, a=a, b=b))
    return pd.DataFrame(rows)


def fold_cka(feats, subj, models, kind_of):
    """feats: {name: {level: (N, d)}} incl. 'random <kind>' entries; subj: (N,) subject ids.
    Returns per-subject rows and the fold-level (all test epochs of the fold) rows."""
    pairs = [(x, y) for x, y in itertools.combinations(models, 2)]
    pairs += [(x, f'random {kind_of[y]}') for x, y in itertools.combinations(models, 2)]
    pairs = list(dict.fromkeys(pairs))
    per_subj, per_fold = [], []
    for lv in LEVELS:
        for x, y in pairs:
            X, Y = feats[x][lv], feats[y][lv]
            per_fold.append(dict(level=lv, pair=f'{x} ~ {y}', cka=linear_cka(X, Y), n_epochs=len(X)))
            for s in np.unique(subj):
                m = subj == s
                per_subj.append(dict(level=lv, pair=f'{x} ~ {y}', subject=s, cka=linear_cka(X[m], Y[m]),
                                     n_epochs=int(m.sum())))
    return pd.DataFrame(per_subj), pd.DataFrame(per_fold)


def run_tests(plan, per_subj, exclude_subjects=()):
    from xai import wilcoxon_row, bh_by_family
    t = per_subj[~per_subj.subject.isin(list(exclude_subjects))]
    P = t.set_index(['level', 'pair', 'subject']).cka
    out = []
    for r in plan.itertuples():
        a, b = P.loc[(r.level, r.a)], P.loc[(r.level, r.b)]
        d = (a - b.reindex(a.index)).dropna()
        out.append(wilcoxon_row(d.values, family=r.family, level=r.level, a=r.a, b=r.b,
                                median_a=float(a.median()), median_b=float(b.median())))
    res = bh_by_family(pd.DataFrame(out))
    return res.rename(columns={'n_excerpts': 'n_subjects'})

In [ ]:
%%writefile /content/skd_code/qsim.py
# qsim.py — PC simulation of the deployed mixed-precision Edge Student (D4 / quantization evaluation).
# Deployed configuration (revision/E_hardware/R2/firmware): the depthwise + pointwise convolutions of the four DWSep
# blocks (EEG 0-1, EOG 0-1) run in INT8 (ESP-DL, quantized with ESP-PPQ); GroupNorm + GELU, pooling, projection, GRU,
# LayerNorms and classifier run in FP32. Nothing else is quantized here.
import copy, inspect, os, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

BLOCKS = [('eeg_block_0', 'eeg_branch', 0), ('eog_block_0', 'eog_branch', 0),
          ('eeg_block_1', 'eeg_branch', 1), ('eog_block_1', 'eog_branch', 1)]
STAGES = ['W', 'N1', 'N2', 'N3', 'REM']


def strict_fp32():
    """No TF32 / non-deterministic kernels: the reference must be genuine FP32."""
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ConvOnly(nn.Module):        # depthwise + pointwise only (GroupNorm and GELU run in FP32 on the chip)
    def __init__(self, blk):
        super().__init__()
        self.dw, self.pw = blk.dw, blk.pw

    def forward(self, x):
        return self.pw(self.dw(x))


def calibration_subset(cal, n_cal=512, seed=42):
    """Same subsample as revision/E_hardware/R2/quantize_export.py (512 of the stratified training epochs)."""
    rng = np.random.default_rng(seed)
    return cal[np.sort(rng.choice(len(cal), min(n_cal, len(cal)), replace=False))]


@torch.no_grad()
def calibration_inputs(model, cal):
    """Input of each block on the calibration epochs; block 1 sees the FP32 output of the full block 0
    (incl. GroupNorm + GELU), as in the deployed export."""
    cnn = copy.deepcopy(model.cnn).cpu().eval()
    eeg, eog = cal[:, :2], cal[:, 2:3]
    return {'eeg_block_0': eeg, 'eog_block_0': eog,
            'eeg_block_1': cnn.eeg_branch[0](torch.from_numpy(eeg)).numpy(),
            'eog_block_1': cnn.eog_branch[0](torch.from_numpy(eog)).numpy()}


def graph_exponents(graph):
    """log2 of every power-of-two scale in a quantized block (input, weights, intermediate, output)."""
    out = {}
    for op in graph.operations.values():
        for cfg, var in op.config_with_variable:
            if cfg.scale is not None:
                e = np.log2(cfg.scale.detach().cpu().numpy().astype(np.float64).ravel())
                assert np.allclose(e, np.round(e)), f'{var.name}: scale is not a power of two'
                out[var.name] = [int(v) for v in np.round(e)]
    return out


def quantize_blocks(model, cal, target, work_dir, export_dir):
    """Quantize the four convolution blocks with ESP-PPQ (INT8, target chip), export the .espdl files to export_dir.
    Returns {block: quantized graph} and per-block info (shapes, exponents, rounding)."""
    from esp_ppq.api import espdl_quantize_onnx
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)
    ins = calibration_inputs(model, cal)
    graphs, info = {}, {}
    for name, branch, i in BLOCKS:
        blk = getattr(model.cnn, branch)[i]
        data = np.ascontiguousarray(ins[name], dtype=np.float32)
        shape = [1] + list(data.shape[1:])
        onnx_path = os.path.join(work_dir, f'{name}.onnx')
        kw = {'dynamo': False} if 'dynamo' in inspect.signature(torch.onnx.export).parameters else {}
        torch.onnx.export(ConvOnly(copy.deepcopy(blk)).cpu().eval(), torch.zeros(shape), onnx_path, input_names=['input'],
                          output_names=['output'], opset_version=13, **kw)
        loader = DataLoader(TensorDataset(torch.from_numpy(data)), batch_size=1, shuffle=False)
        g = espdl_quantize_onnx(onnx_import_file=onnx_path, espdl_export_file=os.path.join(export_dir, f'{name}.espdl'),
                                calib_dataloader=loader, calib_steps=len(data), input_shape=shape, target=target,
                                num_of_bits=8, export_test_values=False, device='cpu', error_report=False)
        for side in ('.info', '.json'):          # ESP-PPQ side files: keep them out of the export folder
            p = os.path.join(export_dir, name + side)
            if os.path.exists(p):
                shutil.move(p, os.path.join(work_dir, f'{name}_{target}{side}'))   # may cross file systems (Drive)
        rounding = sorted({str(cfg.rounding) for op in g.operations.values() for cfg, _ in op.config_with_variable})
        graphs[name] = g
        info[name] = dict(input_shape=shape, groupnorm_groups=blk.norm.num_groups, exponents=graph_exponents(g),
                          rounding=rounding)
    return graphs, info


class QuantEdge:
    """Edge Student with the four convolution blocks executed by the ESP-PPQ simulator of the target chip.
    The block input is quantized like the firmware (round half away from zero, clip to [-128, 127]) before the
    simulator, so the simulator's own input quantization is the identity."""

    def __init__(self, model, graphs, device):
        from esp_ppq.executor import TorchExecutor
        self.m, self.device = model.to(device).eval(), device
        self.ex = {k: TorchExecutor(graph=g, device=str(device)) for k, g in graphs.items()}
        self.in_scale = {}
        for k, g in graphs.items():
            cfg = [c for op in g.operations.values() for c, v in op.config_with_variable if v.name == 'input']
            self.in_scale[k] = float(cfg[0].scale.ravel()[0])

    def _block(self, name, branch, i, x):
        s = self.in_scale[name]
        v = x / s
        q = torch.clamp(torch.sign(v) * torch.floor(torch.abs(v) + 0.5), -128, 127) * s
        y = self.ex[name].forward(inputs=q)[0]
        blk = getattr(self.m.cnn, branch)[i]
        return blk.act(blk.norm(y))

    @torch.no_grad()
    def features(self, x):                 # x: (N, 3, 3000) float32 on device -> (N, 64)
        cnn = self.m.cnn
        eeg = self._block('eeg_block_1', 'eeg_branch', 1, self._block('eeg_block_0', 'eeg_branch', 0, x[:, :2]))
        eog = self._block('eog_block_1', 'eog_branch', 1, self._block('eog_block_0', 'eog_branch', 0, x[:, 2:3]))
        assert eeg.shape[-1] == eog.shape[-1]
        return cnn.proj(cnn.pool(torch.cat([eeg, eog], 1)).squeeze(-1))


def _q_half_up(x, e):                        # ESP-PPQ ROUND_HALF_UP, symmetric INT8, power-of-two scale 2**e
    s = 2.0 ** e
    return torch.clamp(torch.floor(x / s + 0.5), -128, 127) * s


def _q_half_away(x, e):                      # firmware input rounding (roundf: half away from zero)
    s = 2.0 ** e
    v = x / s
    return torch.clamp(torch.sign(v) * torch.floor(torch.abs(v) + 0.5), -128, 127) * s


def check_params(params):
    """params: {block: {'exponents': {var: [e]}, 'rounding': [...]}} as returned by quantize_blocks (info)."""
    for b, v in params.items():
        assert v['rounding'] == ['RoundingPolicy.ROUND_HALF_UP'], (b, v['rounding'])
        ex = v['exponents']
        assert set(ex) == {'input', 'dw.weight', '/dw/Conv_output_0', 'pw.weight', 'output'}, (b, sorted(ex))
        assert all(len(e) == 1 for e in ex.values()), f'{b}: per-channel scales are not supported here'


class TorchQuantEdge:
    """Pure-PyTorch re-implementation of the ESP-PPQ simulation used by QuantEdge (no esp_ppq needed): per block,
    input -> INT8 (firmware rounding), depthwise conv with INT8 weights -> INT8 (half up), pointwise conv with INT8
    weights -> INT8 output (half up); then FP32 GroupNorm + GELU. Verified against QuantEdge (ESP-PPQ executor)."""

    def __init__(self, model, params, device):
        check_params(params)
        self.m, self.device = model.to(device).eval(), device
        self.e = {b: {k: int(v[0]) for k, v in p['exponents'].items()} for b, p in params.items()}
        self.w = {}
        for name, branch, i in BLOCKS:
            blk, e = getattr(self.m.cnn, branch)[i], self.e[name]
            self.w[name] = (_q_half_up(blk.dw.weight.detach(), e['dw.weight']), _q_half_up(blk.pw.weight.detach(), e['pw.weight']))

    def _block(self, name, branch, i, x):
        blk, e = getattr(self.m.cnn, branch)[i], self.e[name]
        wd, wp = self.w[name]
        y = torch.nn.functional.conv1d(_q_half_away(x, e['input']), wd, None, blk.dw.stride, blk.dw.padding,
                                       blk.dw.dilation, blk.dw.groups)
        y = _q_half_up(y, e['/dw/Conv_output_0'])
        y = _q_half_up(torch.nn.functional.conv1d(y, wp, None, blk.pw.stride, blk.pw.padding, blk.pw.dilation,
                                                  blk.pw.groups), e['output'])
        return blk.act(blk.norm(y))

    @torch.no_grad()
    def features(self, x):
        cnn = self.m.cnn
        eeg = self._block('eeg_block_1', 'eeg_branch', 1, self._block('eeg_block_0', 'eeg_branch', 0, x[:, :2]))
        eog = self._block('eog_block_1', 'eog_branch', 1, self._block('eog_block_0', 'eog_branch', 0, x[:, 2:3]))
        assert eeg.shape[-1] == eog.shape[-1]
        return cnn.proj(cnn.pool(torch.cat([eeg, eog], 1)).squeeze(-1))


@torch.no_grad()
def fp32_features(model, x):
    return model.cnn(x)


@torch.no_grad()
def head(model, seq):                       # seq: (B, L, 64) -> logits of the centre epoch (GRU is causal)
    g, _ = model.gru(seq)
    return model.classifier(model.gru_norm(g[:, seq.shape[1] // 2]))


@torch.no_grad()
def causal_head(model, seq):                # seq: (B, 11, 64) -> logits after the last step (as on the chip)
    g, _ = model.gru(seq)
    return model.classifier(model.gru_norm(g[:, -1]))


@torch.no_grad()
def recording_logits(feat_fn, model, x, centers, device, half=10, chunk=1024):
    """Logits for the target epochs `centers` of one recording, from per-epoch features computed once.
    Windows are clamped to the recording (edge-replicate padding), exactly as skd.SeqDS."""
    n = len(x)
    lo, hi = max(0, int(centers.min()) - half), min(n - 1, int(centers.max()) + half)
    feats = torch.empty(hi - lo + 1, model.cnn.proj[0].out_features, device=device)
    for s in range(lo, hi + 1, chunk):
        e = min(hi + 1, s + chunk)
        feats[s - lo:e - lo] = feat_fn(torch.from_numpy(x[s:e].astype(np.float32)).to(device))
    idx = np.clip(centers[:, None] + np.arange(-half, half + 1)[None], 0, n - 1) - lo
    out = []
    for s in range(0, len(idx), chunk):
        out.append(head(model, feats[torch.from_numpy(idx[s:s + chunk]).to(device)]).float().cpu())
    return torch.cat(out).numpy()


# ---------------------------------------------------------------- statistics (paired subject-cluster bootstrap)
def cm_metrics(cm):
    """cm: (..., 5, 5) -> dict of arrays: mf1, acc, kappa, f1_<stage>."""
    cm = cm.astype(np.float64)
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    rows, cols = cm.sum(-1), cm.sum(-2)
    n = cm.sum((-1, -2))
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rows + cols > 0, 2 * tp / np.where(rows + cols > 0, rows + cols, 1), 0.0)
        acc = tp.sum(-1) / n
        pe = (rows * cols).sum(-1) / n ** 2
        kappa = (acc - pe) / (1 - pe)
    out = dict(mf1=f1.mean(-1), acc=acc, kappa=kappa)
    out.update({f'f1_{s}': f1[..., k] for k, s in enumerate(STAGES)})
    return out


def subject_cms(y, pred, subj):
    subs = np.unique(subj)
    cms = np.zeros((len(subs), 5, 5), np.int64)
    for k, s in enumerate(subs):
        m = subj == s
        np.add.at(cms[k], (y[m], pred[m]), 1)
    return subs, cms


def bh(p):
    p = np.asarray(p, float)
    o = np.argsort(p)
    q = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[o] = np.minimum(q, 1.0)
    return out


def paired_bootstrap(cm_a, cm_b, n_boot=10000, seed=0):
    """A - B for every metric; subjects resampled with replacement (same replicate subjects for A and B).
    Returns rows with the point difference (pp), 95% percentile CI and two-sided bootstrap p (+1 correction)."""
    rng = np.random.default_rng(seed)
    S = len(cm_a)
    W = rng.multinomial(S, np.full(S, 1 / S), size=n_boot)
    ma, mb = cm_metrics(cm_a.sum(0)), cm_metrics(cm_b.sum(0))
    ba = cm_metrics(np.einsum('bs,sij->bij', W, cm_a))
    bb = cm_metrics(np.einsum('bs,sij->bij', W, cm_b))
    rows = []
    for met in ma:
        d = ba[met] - bb[met]
        p = min(1.0, 2 * (min((d <= 0).sum(), (d >= 0).sum()) + 1) / (n_boot + 1))
        rows.append(dict(metric=met, family='primary' if met == 'mf1' else 'secondary', value_a=100 * float(ma[met]),
                         value_b=100 * float(mb[met]), diff_pp=100 * float(ma[met] - mb[met]),
                         ci95_low_pp=100 * float(np.percentile(d, 2.5)), ci95_high_pp=100 * float(np.percentile(d, 97.5)),
                         p_boot=p, n_subjects=S, n_epochs=int(cm_a.sum())))
    sec = [r for r in rows if r['family'] == 'secondary']
    for r, q in zip(sec, bh([r['p_boot'] for r in sec])):
        r['q_bh'] = float(q)
    rows[0]['q_bh'] = rows[0]['p_boot']           # single primary test
    return rows


# ---------------------------------------------------------------- CLI (run with a Python that has esp-ppq, e.g. 3.12)
def _cli():
    """python qsim.py quantize --code-dir D --model best.pt --target-dim N --calib cal.npy --targets esp32 esp32s3
                              --work W --export E --out params.json [--check check.npy]
    Quantizes the four blocks for each target (ESP-PPQ), writes {target: {block: info}} to --out and, with --check,
    the ESP-PPQ-executor features of those epochs (CPU) to <out>.check_<target>.npy for comparison with TorchQuantEdge."""
    import argparse, json, sys
    from importlib.metadata import version
    ap = argparse.ArgumentParser()
    ap.add_argument('cmd', choices=['quantize'])
    ap.add_argument('--code-dir', required=True)
    ap.add_argument('--model', required=True)
    ap.add_argument('--target-dim', type=int, default=0)
    ap.add_argument('--calib', required=True)
    ap.add_argument('--targets', nargs='+', required=True)
    ap.add_argument('--work', required=True)
    ap.add_argument('--export', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--check', default=None)
    a = ap.parse_args()
    sys.path.insert(0, a.code_dir)
    from model_v11.student_v2 import StudentSleepNetV2
    strict_fp32()
    model = StudentSleepNetV2(embed_dim=64, gru_hidden=32, gru_layers=1, v1_embed_dim=a.target_dim or 128,
                              num_classes=5, seq_len=21, dropout=0.1)
    model.load_state_dict(torch.load(a.model, map_location='cpu', weights_only=True))
    model.eval()
    cal = np.load(a.calib).astype(np.float32)
    out = dict(esp_ppq=version('esp-ppq'), onnx=version('onnx'), torch=torch.__version__, n_calibration=len(cal))
    for t in a.targets:
        graphs, info = quantize_blocks(model, cal, t, os.path.join(a.work, t), os.path.join(a.export, t))
        check_params(info)
        out[t] = info
        if a.check:
            x = torch.from_numpy(np.load(a.check).astype(np.float32))
            np.save(f'{a.out}.check_{t}.npy', QuantEdge(model, graphs, torch.device('cpu')).features(x).numpy())
    json.dump(out, open(a.out, 'w'), indent=1)
    print('quantized:', a.targets, '| esp-ppq', out['esp_ppq'], '| onnx', out['onnx'], '| torch', out['torch'])


if __name__ == '__main__':
    _cli()

In [ ]:
import sys, json, importlib, shutil, time
sys.path.insert(0, CODE_DIR)
import torch, numpy as np, pandas as pd
import prep, skd
importlib.reload(prep); importlib.reload(skd)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '',
      '| bf16:', device.type == 'cuda' and torch.cuda.is_bf16_supported())

## 1. Nạp dữ liệu và kiểm tra nguồn distill seed 42

In [ ]:
t0 = time.time()
shutil.copytree(DATA_DIR, LOCAL_DATA, dirs_exist_ok=True)
folds = {int(k): v for k, v in json.load(open(FOLDS_JSON)).items()}
store = skd.Store(LOCAL_DATA, sorted(r for v in folds.values() for r in v))
for src in ('T_ce', 'A_kd_Tce'):
    for f in range(K_FOLDS):
        d = skd.run_dir(dict(RUNS_DIR=RUNS_DIR), src, f, 42)
        assert os.path.exists(f'{d}/best.pt') and os.path.exists(f'{d}/result.json'), f'missing {src} fold{f} seed42'
print(f'{len(store.recs)} recordings in RAM | nguồn seed 42 đủ 5 fold | {time.time() - t0:.0f}s')

## 2. Train (resume được; mỗi phiên chạy các seed trong `RUN_SEEDS`)

In [ ]:
skd.run_all(C_NEW, store, folds, device, run_folds=RUN_FOLDS)

## 3. Phân tích (chạy khi đủ cả seed 43 và 44)

In [ ]:
f1_subjects = sorted({r[:5] for r in folds[1]})
arms = [a['name'] for a in SEED_ARMS]
missing = [(a, s, f) for a in arms for s in [42] + NEW_SEEDS for f in range(K_FOLDS)
           if skd.load_result(dict(RUNS_DIR=RUNS_DIR), a, f, s) is None]
assert not missing, f'chưa xong: {missing}'
for a in arms:                       # same protocol as seed 42; distillation source = seed-42 model of the same fold
    for f in range(K_FOLDS):
        r42 = skd.load_result(dict(RUNS_DIR=RUNS_DIR), a, f, 42)
        for s in NEW_SEEDS:
            r = skd.load_result(dict(RUNS_DIR=RUNS_DIR), a, f, s)
            assert all(r[k] == r42[k] for k in ('kind', 'loss', 'src', 'T', 'lr', 'ls', 'bs', 'n_train', 'n_test')), (a, s, f)
            assert r['src_seed'] == (42 if r['src'] else None), (a, s, f)
print('giao thức seed 43/44 == seed 42; nguồn distill = model seed 42 cùng fold')
C_ALL = dict(TRAIN_CFG, K_FOLDS=K_FOLDS, RUNS_DIR=RUNS_DIR, SEEDS=[42] + NEW_SEEDS)
per_seed = pd.concat([skd.pooled_table(dict(C_ALL, SEEDS=[s]), arms, exclude_subjects=f1_subjects).assign(seed=s)
                      for s in [42] + NEW_SEEDS])
per_seed.to_csv(f'{RESULTS}/seed_robustness_pooled_per_seed.csv')
display(per_seed[['seed', 'mf1', 'acc', 'kappa', 'f1_N1', 'n_subjects']].round(2))
ct79 = skd.cluster_contrasts(C_ALL, SEED_CONTRASTS, n_boot=10000, exclude_subjects=f1_subjects).assign(subjects=79, seeds='42+43+44')
ct100 = skd.cluster_contrasts(C_ALL, SEED_CONTRASTS, n_boot=10000).assign(subjects=100, seeds='42+43+44')
cts = [skd.cluster_contrasts(dict(C_ALL, SEEDS=[s]), SEED_CONTRASTS, n_boot=10000, exclude_subjects=f1_subjects)
       .assign(subjects=79, seeds=str(s)) for s in [42] + NEW_SEEDS]
allc = pd.concat([ct79, ct100] + cts)
allc.to_csv(f'{RESULTS}/seed_robustness_contrasts.csv', index=False)
allc[allc.metric.isin(['mf1', 'n1'])][['seeds', 'subjects', 'contrast', 'metric', 'diff_pp', 'ci95_low_pp', 'ci95_high_pp', 'q_bh']].round(4)